# The two modelling choices: priors and likelihood

DALEC2 calibrated against daily net ecosystem exchange at FI-Hyy (Hyytiälä, boreal Scots pine). Master's thesis, Tampere University.

This notebook explains **two inputs to the inference and nothing else**: the prior, which says what parameter values are plausible before any data is seen, and the likelihood, which says how probable an observed flux is for a given parameter set. It is not about sampling results on real data, and it contains none.

**Nothing here has been run.** Every output cell is empty by design and no figure is written to disk. The cells are small and ordered, so they can be run one at a time.

### How to read it

| Part | Sections | Question it answers |
|---|---|---|
| Setup | 0-2 | What is in scope, and where do the prior and the likelihood sit in the model? |
| A | 3-6 | Where did the priors come from, and are they physically coherent? |
| B | 7-11 | What does the likelihood assume, and what does each assumption cost if it is wrong? |
| Closing | 12-13 | Talking points, open questions, and a critique of this notebook itself |

### Terms, each defined once where it first appears

- **Parameter**: one of the 24 numbers DALEC2 needs, such as a turnover rate or an allocation fraction.
- **Prior**: the distribution of a parameter before any data is seen.
- **Likelihood**: the probability of the observed data, read as a function of the parameters.
- **Posterior**: the distribution of the parameters after conditioning on the data.
- **Draw**: one sampled parameter set, all 24 numbers at once.
- **NUTS**: the sampler in use. It follows the gradient of the log posterior, which is why the forward model is written in PyTensor and why any hard accept/reject rule is a problem (section 6).

---

# 0. Config

Everything adjustable lives in the next five cells and nowhere else. No path, threshold, seed or count is retyped further down.

One rule carried over from the codebase: **no prior bound is typed in this notebook.** Bounds are read from `dalec.priors.prior_sources`, `dalec.parameters.canopy_bounds` and `dalec.diagnostics.reparameterised_bounds`, which in turn read them from their recorded sources. What is typed here is the provenance metadata -- symbol, unit, citation, category -- which exists only as prose elsewhere.

In [ ]:
# Standard library, third party, and the repository root.
#
# The root is found by walking up from the working directory until
# config/default.yaml appears, so this runs the same whether Jupyter was
# started in notebooks/ or at the repository root.
import dataclasses
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'config' / 'default.yaml').is_file():
            return candidate
    raise FileNotFoundError(
        'could not find the repository root: no config/default.yaml in this '
        'directory or any parent of ' + str(here)
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('repository root:', REPO_ROOT)
print('python:', sys.version.split()[0])

: 

In [ ]:
# Project imports. Every name here is public API of src/dalec.
from dalec.acm import acm_from_config, leaf_area_index
from dalec.config import load_config, require_year_block, resolve_path
from dalec.data_io import SiteData
from dalec.diagnostics import (
    FAILURE_GROWTH_FACTOR,
    FAILURE_NEE_LIMIT,
    reparameterised_bounds,
    sample_reparameterised_parameters,
)
from dalec.model_numpy import POOL_NAMES, run_dalec2
from dalec.parameters import (
    ALLOCATION_PARAMETERS,
    ALLOCATION_WEIGHT_ORDER,
    PARAMETER_NAMES,
    PARAMETER_REGISTRY,
    DalecParameters,
    allocation_concentration,
    canopy_bounds,
    site_lai,
)
from dalec.plotting import OKABE_ITO, apply_style
from dalec.priors import prior_sources

print('model parameters:', len(PARAMETER_NAMES))
print('carbon pools    :', ', '.join(POOL_NAMES))

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 1 of 5 -- scope switches
#
# These decide how much of the real FI-Hyy record this notebook touches.
# Section 1 states what each one means and what turning it off would cost.
# ---------------------------------------------------------------------------

# True  : drivers are the real FI-Hyy meteorological record.
# False : drivers are smooth idealised substitutes generated in this notebook.
USE_REAL_DRIVERS = True

# True  : the per-day noise scale is the real NEE_VUT_REF_RANDUNC column.
# False : a single constant noise scale, IDEALISED_SIGMA, on every day.
USE_REAL_SIGMA = True
IDEALISED_SIGMA = 0.5          # g C m-2 d-1, used only when USE_REAL_SIGMA is False

# True  : the set of days that enter the likelihood is the real QC mask.
# False : every day enters.
USE_REAL_DAY_MASK = True

# No observed NEE flux value is read under any setting. This is asserted in
# code at the end of the config block, not merely promised here.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 2 of 5 -- paths, block, seeds
# ---------------------------------------------------------------------------
CONFIG_PATH = REPO_ROOT / 'config' / 'default.yaml'

# LAI convention. Both stay live in this project and neither is adopted; it
# changes lma and nothing else. 'hemisurface' or 'projected'.
CONVENTION = 'hemisurface'

# Inclusive year range this notebook runs over. One year keeps the first pass
# cheap; cost is linear in the number of days.
DRIVER_YEARS = (1998, 1998)

# The year drawn in the prior predictive fan plot. Must be inside DRIVER_YEARS.
FAN_YEAR = 1998

# Seeds. Kept separate on purpose: the true parameter set must not be one of
# the prior predictive draws, or the check would be scored against a member of
# its own sample.
SEED_PRIOR_DRAWS = 20260809       # matches config/default.yaml
SEED_TRUTH = 90210                # the parameter set the synthetic data comes from
SEED_NOISE = 424242               # the noise added to the synthetic data

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 3 of 5 -- prior predictive check (Part A)
# ---------------------------------------------------------------------------

# Prior draws. Start at 200; raise once the machinery is checked.
# NOTE: the draws are NOT nested. Each parameter is drawn as one vector of
# length N_PRIOR_DRAWS, so changing this number changes every draw, not just
# how many there are. 200 and 1000 are two samples of the same prior, not a
# subset and a superset.
N_PRIOR_DRAWS = 200

# Progress line every this many draws.
PROGRESS_EVERY = 50

# --- physical plausibility thresholds --------------------------------------
# Section 5c states the justification for each of these next to the rule.

GPP_FLOOR_G_C_M2_D = 0.0       # definitional: gross uptake cannot be negative

# LAI ceiling as a multiple of the site's measured seasonal maximum on the
# chosen basis. The site value is sourced; the multiple is a stated tolerance.
LAI_CEILING_MULTIPLE = 2.0

# Plausible annual NEE window, g C m-2 yr-1, negative being net uptake.
# TODO: this is a STATED ASSUMPTION with no source. See section 5c for why the
# site's own measured annual NEE is deliberately not used as the bound, and
# section 12 for the question that raises for the meeting.
ANNUAL_NEE_RANGE_G_C_M2_YR = (-600.0, 300.0)

# A calendar year needs at least this many days in the block to be summed.
MIN_DAYS_FOR_ANNUAL = 365

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 4 of 5 -- likelihood experiments (Part B)
# ---------------------------------------------------------------------------

# --- the one-parameter likelihood scan used for the section 8 demonstrations
# A cheap stand-in for a full fit: hold every parameter at its true value,
# vary one, and look at the log-likelihood along that line.
SCAN_PARAMETER = 'ceff'        # canopy efficiency; scales GPP
SCAN_POINTS = 41               # forward-model runs, so this is the cost driver
SCAN_SPAN = 0.6                # fraction of the prior range to scan, centred on truth

# --- assumption demonstrations
HEAVY_TAIL_NU = 3.0            # degrees of freedom of the heavy-tailed noise
SIGMA_SCALES = (0.5, 0.71, 1.0, 1.41, 2.0)   # uniform misstatements of sigma
SUMMER_SIGMA_INFLATION = 2.0   # a structured, seasonal misstatement of sigma
SUMMER_MONTHS = (6, 7, 8)
AR1_RHO = 0.6                  # day-to-day correlation of the noise
N_NOISE_REPLICATES = 400       # replicates for the independence demonstration
BIAS_VALUES = (-0.4, -0.2, 0.0, 0.2, 0.4)    # constant offsets, g C m-2 d-1

# --- section 9, the misspecification experiment: the only cells that sample
# Set to False for a first pass that skips the two NUTS fits entirely; the
# table and figure then print a message instead of failing.
RUN_MISSPECIFICATION_FITS = True

SAMPLE_DRAWS = 200
SAMPLE_TUNE = 200
SAMPLE_CHAINS = 2
SAMPLE_CORES = 1               # 1 keeps chains sequential, which is the reliable
                               # setting inside a notebook on Windows
FIT_NU = 4.0                   # degrees of freedom of the heavier-tailed likelihood

# Match the Student-t scale so that its variance equals sigma^2 rather than
# sigma^2 * nu/(nu-2). See section 9 for why this matters.
MATCH_STUDENT_T_VARIANCE = True

# Noise replicates for the coverage column. 1 gives coverage across parameters
# for a single dataset, which is NOT frequentist coverage; see section 9.
N_COVERAGE_REPLICATES = 1

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 5 of 5 -- load the project config and resolve the derived settings
# ---------------------------------------------------------------------------
apply_style()          # the shared project figure style, src/dalec/plotting.py
warnings.filterwarnings('ignore', category=RuntimeWarning)

CONFIG = load_config(CONFIG_PATH)

CALIBRATION_YEARS = require_year_block(CONFIG, 'calibration')
HDI_PROB = float(CONFIG['diagnostics']['hdi_prob'])
SITE_CODE = str(CONFIG['site']['code'])
SITE_SLUG = SITE_CODE.lower().replace('-', '_')

PROCESSED_DIR = resolve_path(CONFIG['paths']['processed_dir'])
CALIBRATION_BLOCK_PATH = (
    PROCESSED_DIR
    / (SITE_SLUG + '_calibration_' + str(CALIBRATION_YEARS[0])
       + '_' + str(CALIBRATION_YEARS[1]) + '.nc')
)

SITE_LAI_AFTER, SITE_LAI_BEFORE = site_lai(CONVENTION)
LAI_CEILING = LAI_CEILING_MULTIPLE * SITE_LAI_BEFORE

# One palette for the whole notebook, so the figures read as a set.
COLOUR = {
    'prior': OKABE_ITO[0],        # blue
    'truth': OKABE_ITO[2],        # bluish green
    'gaussian': OKABE_ITO[0],     # blue
    'heavy': OKABE_ITO[1],        # vermillion
    'neutral': OKABE_ITO[7],      # black
    'accent': OKABE_ITO[4],       # orange
}
FIGSIZE_WIDE = (10.5, 4.6)
FIGSIZE_GRID = (3.3, 2.4)        # per panel

print('site            ', SITE_CODE, '|', CONFIG['site']['name'])
print('convention      ', CONVENTION)
print('block           ', DRIVER_YEARS)
print('credible level  ', HDI_PROB)
print('site LAI max    ', round(SITE_LAI_BEFORE, 3), 'before thinning,',
      round(SITE_LAI_AFTER, 3), 'after')
print('LAI ceiling     ', round(LAI_CEILING, 3))
print()
print('scope switches   real drivers', USE_REAL_DRIVERS,
      '| real sigma', USE_REAL_SIGMA, '| real day mask', USE_REAL_DAY_MASK)

---

# 1. Scope statement

This section exists so that the boundary of the notebook can be agreed, or objected to, one line at a time.

### (a) No observed NEE is used anywhere in this notebook.

Not in a fit, not in a plot, not in a table, not as a reference band. Every flux value shown is produced by the forward model from a parameter set, and the only "truth" anywhere is the parameter set the synthetic data was generated from. The cell after this section asserts it in code rather than leaving it as a promise.

*Why:* the purpose here is to examine the prior and the likelihood as modelling choices. Once real fluxes enter, any statement about them becomes a statement about the fit, and that is a different notebook.

### (b) The meteorological drivers are the real FI-Hyy record.

Daily mean, maximum and minimum air temperature, incoming shortwave radiation and atmospheric CO2, taken from the processed calibration block. The forward model cannot be run without weather, and these are **model inputs, not calibration targets**: no parameter is fitted to them and they appear nowhere in the likelihood.

*Switch:* `USE_REAL_DRIVERS = False` replaces them with smooth sinusoidal substitutes generated here.
*What would be lost:* the real record's weather sequence -- cold snaps, cloudy weeks, the shape of the spring warming -- is what makes some days informative and others not. Idealised drivers would make every year identical and would remove the very irregularity that section 10 is about.

### (c) The per-day noise scale is taken from the real random-uncertainty column.

The synthetic data is built as model output plus noise, and the standard deviation of that noise on each day is `NEE_VUT_REF_RANDUNC` for that day. So the *structure* of the noise -- how much larger it is in summer than in winter -- is borrowed from the site, even though no flux value is.

*Switch:* `USE_REAL_SIGMA = False` uses one constant standard deviation, `IDEALISED_SIGMA`, on every day.
*What would be lost:* the seasonal concentration of information. Under a constant sigma every day carries equal weight, which is exactly the thing section 10 measures and would make that section vacuous.

### (d) The set of days that enters the likelihood is the real quality-control mask. *(my addition -- please object if you disagree)*

A day enters only if the real record had an observation there that passed the quality threshold and carried a usable uncertainty. That is information about **which days were measured**, not about what was measured.

*Switch:* `USE_REAL_DAY_MASK = False` lets every day contribute.
*What would be lost:* the real pattern of gaps, which is itself seasonal at a boreal site, so an all-days mask would quietly change the seasonal balance of the fit.

### What (b), (c) and (d) have in common

All three borrow *structure* from the site while borrowing no *flux value*. If you would rather the notebook borrowed none of it, the three switches in the config cell turn them off together and the notebook still runs end to end.

In [ ]:
# The scope assertion. This is a guard, not a formality: it fails loudly if a
# later edit reads the observed flux column.
#
# SiteData.nee_obs is the observed NEE series. Nothing in this notebook may
# read it, so it is replaced with NaN on the block the whole notebook uses and
# the replacement is checked after the block is built (next section).
OBSERVED_FLUX_FIELD = 'nee_obs'


def assert_no_observed_flux(block, where):
    """Raise unless the observed NEE series on this block is entirely absent."""
    values = getattr(block, OBSERVED_FLUX_FIELD)
    if np.isfinite(values).any():
        raise AssertionError(
            'scope violation at ' + where + ': ' + str(int(np.isfinite(values).sum()))
            + ' finite observed NEE values are present on the block. Section 1(a) '
            'says none may be.'
        )
    return True


print('scope guard defined; it is called after the driver block is built')

---

# 2. Where the prior and the likelihood sit in the model

Bayes' rule, for a parameter set $\theta$ and observations $y$:

$$p(\theta \mid y) \;\propto\; p(y \mid \theta)\; p(\theta)$$

*In words: how plausible a parameter set is after seeing the data equals how well it explains the data, multiplied by how plausible it was beforehand.*

The likelihood, written out over the days that enter the fit:

$$p(y \mid \theta) \;=\; \prod_{d \in D} \mathcal{N}\!\left(y_d \;\middle|\; m_d(\theta),\; \sigma_d^2\right)$$

*In words: every included day contributes one factor, saying how surprising that day's observation is given what the model predicts for it and how uncertain that day's observation is.*

$m_d(\theta)$ is the model's predicted NEE on day $d$. It is not a formula in $\theta$: it is the output of a 365-step integration of six carbon pools, which is why the whole thing has to be differentiable.

### How the pieces connect

```
        prior  p(theta)                              data  y
        "what is plausible                           (synthetic throughout
         before any data"                             this notebook)
              |                                            |
              v                                            |
        parameters theta                                   |
              |                                            |
              v                                            |
    +---------------------------+                          |
    |  forward model            |                          |
    |  DALEC2, 6 carbon pools,  |                          |
    |  stepped daily, written   |                          |
    |  in PyTensor so that the  |                          |
    |  gradient is available    |                          |
    +---------------------------+                          |
              |                                            |
              v                                            v
        predicted NEE  m(theta)  ------>  likelihood  p(y | theta)
                                          compares m(theta) to y day by
                                          day, each day scaled by sigma_d
                                                     |
                                                     v
                                          posterior  p(theta | y)
                                          explored by NUTS, which needs
                                          d/d(theta) of log p(theta | y)
```

Two things follow from that last line, and both matter later:

1. Anything that makes the log posterior non-differentiable -- a hard accept/reject rule, for example -- breaks the sampler rather than merely slowing it. Section 6 returns to this.
2. The prior and the likelihood enter symmetrically as *terms in a sum* once logged. Neither is a correction to the other; the posterior is their product.

---
---

# PART A: THE PRIORS

---

# 3. How the priors reached their current form

The table below is an **audit trail**, not a list of mistakes. Each row is a version of the prior, what it did, the measurement that showed it doing that, and what replaced it. Every entry points at a file in this repository that holds the evidence, so each claim can be checked rather than taken on trust.

The reason to show it rather than present only the final priors: a prior that has been revised in response to measurements is a stronger object than one that was right first time, provided the revisions are recorded and none of them was made by looking at the fit. None of these was.

In [ ]:
# The audit trail. Every number here is quoted from the file in the evidence
# column; nothing is recomputed in this notebook.
PRIOR_HISTORY = pd.DataFrame([
    {
        'version': 'v1 published',
        'what the priors were':
            'Bounded uniform over the published DALEC2 ranges for all 23 free '
            'parameters, including both soil pools and both decomposition rates.',
        'what went wrong':
            'Physically incoherent fluxes. 837 of 1000 prior draws were unusable, '
            'and the prior median implied about 50 g C m-2 d-1 of soil respiration '
            'at a site whose entire net exchange is of order 0.6.',
        'evidence':
            'reports/prior_diagnostics/FINDINGS_task1.md, failed_draws.csv',
        'what changed':
            'Nothing yet -- this is the measurement that motivated everything below.',
        'what that fixed': '-',
    },
    {
        'version': 'v2 respiration reparameterised',
        'what the priors were':
            'theta_som and c_som_0 sampled independently over their published '
            'ranges, so the prior was placed on two stocks whose product is a flux.',
        'what went wrong':
            'The two ranked first and second in the one-at-a-time sensitivity '
            'screen while being individually meaningless: only their product is '
            'a physical quantity.',
        'evidence':
            'reports/prior_diagnostics/FINDINGS_task2.md, oaat_metrics.csv',
        'what changed':
            'Sample the annual respiration total rh_annual and the SOM share '
            'f_som; derive c_lit_0 and c_som_0 from them. theta_lit and theta_som '
            'take residence-time bounds instead of the published rate bounds.',
        'what that fixed':
            'A draw can no longer place 10^5 g C m-2 of soil carbon behind a fast '
            'turnover rate. DECISIONS.md section 7.',
    },
    {
        'version': 'v3 allocation on a simplex',
        'what the priors were':
            'The allocation fractions sampled independently, with one taken by '
            'subtraction so that they would sum to one.',
        'what went wrong':
            'The subtracted fraction can go negative, which is a hard rejection '
            'boundary. NUTS follows gradients and cannot differentiate through a '
            'wall, so it stalls against it and produces divergences.',
        'evidence':
            'src/dalec/parameters.py, allocation_fractions docstring',
        'what changed':
            'A Dirichlet prior on the 4-simplex, scaled by (1 - f_auto), so every '
            'point in the unconstrained sampling space maps to a valid allocation.',
        'what that fixed':
            'The constraint is satisfied by construction rather than by rejection.',
    },
    {
        'version': 'v4 canopy from site measurements',
        'what the priors were':
            'lma U(10, 400) g C m-2 and c_lf U(0.125, 1.0), the published ranges, '
            'with a flat Dirichlet on allocation.',
        'what went wrong':
            'The flat Dirichlet sent 24.0 per cent of GPP to foliage and labile '
            'where the measured needle litterfall implies 15.0 per cent, and the '
            'resulting feedback ran modelled LAI to a 95th percentile of 48 '
            'against a site canopy near 3.',
        'evidence':
            'reports/prior_diagnostics/dirichlet_concentration.txt, DECISIONS.md section 8',
        'what changed':
            'Dirichlet concentration derived from the measured allocation fluxes; '
            'lma and c_lf bounds derived from measured needle litterfall, needle '
            'longevity and the site LAI.',
        'what that fixed':
            'The canopy is set by what was measured at the site rather than by a '
            'range wide enough to include needles thinner than any conifer grows.',
    },
    {
        'version': 'v5 ceff from REFLEX',
        'what the priors were': 'ceff U(10, 100), the published DALEC2 range.',
        'what went wrong':
            'Recorded as a change of published prior rather than as a failure: the '
            'REFLEX intercomparison publishes a narrower range for the same ACM '
            'parameter, and it extends below the DALEC2 range rather than merely '
            'narrowing it.',
        'evidence': 'DECISIONS.md section 9',
        'what changed': 'ceff U(5, 20).',
        'what that fixed':
            'Adopts a published prior for the parameter as ACM uses it. Note this '
            'was NOT chosen by looking at modelled GPP.',
    },
    {
        'version': 'v6 every initial pool derived',
        'what the priors were':
            'All six initial carbon pools sampled over published uniform ranges.',
        'what went wrong':
            'c_fol_0 U(20, 2000) has median 1010 against the 462-770 the canopy '
            'priors imply, so the modelled foliage never reached steady state '
            'inside the window and every calibration year was a relaxation '
            'transient from a wrong initial state.',
        'evidence': 'src/dalec/priors.py module docstring, DECISIONS.md section 9',
        'what changed':
            'Every initial pool derived from a measured flux and a turnover rate.',
        'what that fixed':
            'No initial state is left on a generic published range.',
    },
    {
        'version': 'v7 rh_ref per draw',
        'what the priors were':
            'The reference respiration rate inverted from the annual total at one '
            'fixed temperature sensitivity.',
        'what went wrong':
            'temperature_exponent is itself sampled, so the realised annual '
            'respiration drifted off target for every draw whose value differed: '
            'median realised annual Rh 299.8 against an intended 280.',
        'evidence': 'src/dalec/parameters.py, reference_respiration_rate docstring',
        'what changed':
            "rh_ref computed from each draw's own temperature sensitivity.",
        'what that fixed':
            'Every draw respires exactly its sampled annual total by construction.',
    },
    {
        'version': 'context: ACM source corrected',
        'what the priors were':
            'Not a prior change. Recorded here because it re-set the evidence '
            'behind the rows above.',
        'what went wrong':
            'Photosynthesis had been implemented from Williams et al. (1997), whose '
            'day-length term contains no latitude and overstates midwinter at 61.85 N '
            'by about 12x. Measurements of prior behaviour made against it are not '
            'transferable.',
        'evidence': 'src/dalec/acm.py, README.md, DECISIONS.md section 5a',
        'what changed':
            'ACM reimplemented from Chuter et al. (2015), the form DALEC implements.',
        'what that fixed':
            'The GPP magnitude gate went from fail to pass. Any prior diagnostic '
            'measured before this must be re-measured, not carried over.',
    },
    {
        'version': '# TODO',
        'what the priors were': '# TODO: fill in any version I have missed',
        'what went wrong': '# TODO',
        'evidence': '# TODO',
        'what changed': '# TODO',
        'what that fixed': '# TODO',
    },
])

print('versions recorded:', len(PRIOR_HISTORY))
PRIOR_HISTORY[['version', 'evidence']]

In [ ]:
# The same table in full. Printed rather than displayed, because the cells are
# paragraphs and a dataframe truncates them.
for _, row in PRIOR_HISTORY.iterrows():
    print('=' * 78)
    print(row['version'].upper())
    print('=' * 78)
    print('  priors were  :', row['what the priors were'])
    print('  went wrong   :', row['what went wrong'])
    print('  evidence     :', row['evidence'])
    print('  changed to   :', row['what changed'])
    print('  that fixed   :', row['what that fixed'])
    print()

### What this table is and is not

**It is an audit trail.** Each revision was made in response to a measurement that is recorded in the repository, and each measurement can be re-run. The point is not that the first attempt was wrong; it is that the priors now in use are the ones that survived a specific set of checks, and the checks are named.

**None of these revisions was made by looking at the fit.** Every one comes from a prior predictive check, a sensitivity screen, or a comparison against a site measurement -- all of which are computable before any calibration is run. A prior tuned until the posterior looked good would not be a prior.

**Two of them are changes of published prior, not corrections of an error.** v5 replaces one published range with another published range for the same quantity. v2's residence-time bounds are likewise a different reading of the literature rather than a repair.

**# TODO:** the last row is left blank deliberately. If there is a version of the priors I have missed, it belongs there rather than being smoothed over.

---

# 4. The priors in use

One row per parameter. The numeric bounds are read from the code; the symbol, unit, citation and provenance are typed in this notebook because they are recorded in prose and in module docstrings rather than in a machine-readable table.

**Provenance takes exactly one of four values**, and the table is sorted so the weakest comes first:

1. `guess` -- a judgement with no source behind it.
2. `generic DALEC` -- a published DALEC-family range, not specific to this site or species.
3. `published for Scots pine or boreal` -- published, and for this species or this biome.
4. `site measurement` -- measured at FI-Hyy or SMEAR II, or derived arithmetically from such a measurement.

Where a source cannot be pointed to, the source column reads `UNVERIFIED` and carries a `# TODO`.

**Not every parameter is drawn.** Three groups are constructed instead:

- The four allocation fractions come from a Dirichlet on a 4-simplex, scaled by $1 - f_{\mathrm{auto}}$: $(f_{\mathrm{lab}}, f_{\mathrm{fol}}, f_{\mathrm{roo}}, f_{\mathrm{woo}}) = (1 - f_{\mathrm{auto}})\,w$ with $\sum_i w_i = 1$. *In words: whatever GPP is not respired away is split four ways, and the split is drawn as a set of proportions, so it can never go negative.*
- All six initial carbon pools are derived from a measured flux and a turnover rate. For foliage: $C_{\mathrm{fol}}(0) = \text{litterfall} / c_{\mathrm{lf}}$. *In words: at steady state a pool holds however much is needed for its measured annual loss to match its measured annual input.*
- Heterotrophic respiration is reparameterised: the annual total and the soil share are sampled, and the two stocks follow.

In [ ]:
# Bounds, read from their recorded sources. No number is typed here.
SAMPLED_BOUNDS = prior_sources(CONVENTION)        # every sampled scalar
CANOPY_BOUNDS = canopy_bounds(CONVENTION)         # lma, c_lf, ceff overrides
RESPIRATION_BOUNDS = reparameterised_bounds()     # rh_annual, f_som, two rates
DIRICHLET_CONCENTRATION = allocation_concentration()

print('sampled scalars:', len(SAMPLED_BOUNDS))
print('Dirichlet concentration over', ALLOCATION_WEIGHT_ORDER)
print(' ', np.round(DIRICHLET_CONCENTRATION, 3),
      ' summing to', round(float(DIRICHLET_CONCENTRATION.sum()), 3))

In [ ]:
# Provenance metadata, typed here because it exists only as prose elsewhere.
# Sources are as recorded in DECISIONS.md and in the src/dalec docstrings.
# Where the repository records no citation, the source reads UNVERIFIED.

BW2015 = 'Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1'
ILV2009 = 'Ilvesniemi et al. (2009), Boreal Environment Research 14(4), 731-753, Fig. 6'

PRIOR_PROVENANCE = {
    'f_som': dict(
        symbol='f_som', role='sampled',
        motivation='SOM share of heterotrophic respiration taken as 0.5-0.9',
        source='UNVERIFIED -- src/dalec/parameters.py states this as a judgement, '
               'not a measurement. # TODO',
        provenance='guess'),
    'f_auto': dict(
        symbol='f_aut', role='sampled',
        motivation='published range 0.3-0.7; the site fluxes independently imply '
                   'about 0.51, which is a consistency check rather than the source',
        source=BW2015, provenance='generic DALEC'),
    'theta_roo': dict(
        symbol='theta_roo', role='sampled',
        motivation='published range', source=BW2015, provenance='generic DALEC'),
    'theta_woo': dict(
        symbol='theta_woo', role='sampled',
        motivation='published range', source=BW2015, provenance='generic DALEC'),
    'theta_min': dict(
        symbol='theta_min', role='sampled',
        motivation='published range; a litter-to-SOM transfer, not a respiratory loss',
        source=BW2015, provenance='generic DALEC'),
    'temperature_exponent': dict(
        symbol='Theta', role='sampled',
        motivation='published range; the site winter flux record gives 0.0366 as a '
                   'reporting reference only',
        source=BW2015, provenance='generic DALEC'),
    'd_onset': dict(
        symbol='d_onset', role='sampled',
        motivation='published range, the whole year', source=BW2015,
        provenance='generic DALEC'),
    'cr_onset': dict(
        symbol='c_ronset', role='sampled',
        motivation='published range', source=BW2015, provenance='generic DALEC'),
    'd_fall': dict(
        symbol='d_fall', role='sampled',
        motivation='published range. NOTE: inert as transcribed -- see README, '
                   'test_d_fall_is_inert_as_transcribed',
        source=BW2015, provenance='generic DALEC'),
    'cr_fall': dict(
        symbol='c_rfall', role='sampled',
        motivation='published range', source=BW2015, provenance='generic DALEC'),
    'ceff': dict(
        symbol='a1', role='sampled',
        motivation='published range for the same ACM parameter in the REFLEX '
                   'intercomparison, adopted in place of the wider DALEC2 U(10, 100)',
        source='Fox et al. (2009), Agricultural and Forest Meteorology 149, '
               '1597-1615, Table 4. DECISIONS.md records this paper as not read '
               'in full. # TODO: confirm Table 4 and the site type',
        provenance='generic DALEC'),
    'c_lf': dict(
        symbol='c_lf', role='sampled',
        motivation='reciprocal of a 3-5 year needle longevity for Scots pine in '
                   'southern Finland',
        source='UNVERIFIED -- the longevity is asserted in DECISIONS.md section 8 '
               'with no citation. Candidate: Kolari et al. (2009), Boreal '
               'Environment Research 14, 761-783, recorded as giving 25 per cent '
               'foliage turnover. # TODO',
        provenance='published for Scots pine or boreal'),
    'theta_lit': dict(
        symbol='theta_lit', role='sampled',
        motivation='litter residence time 1-5 yr against respiration alone, '
                   'bracketed by the Yasso07 AWEN rates 0.66, 4.3, 0.35, 0.22 a-1',
        source='Tuomi et al. (2009), Ecological Modelling 220, 3362-3371, Table 1',
        provenance='published for Scots pine or boreal'),
    'lma': dict(
        symbol='LMA', role='sampled',
        motivation='mean needle litterfall 154 g C m-2 yr-1 times longevity, '
                   'divided by the site LAI on the chosen basis',
        source='litterfall ' + ILV2009 + '; LAI Kolari (2010), Dissertationes '
               'Forestales 99, doi 10.14214/df.99, all-sided seasonal maximum '
               '6.5-8.0; basis conversion Chen and Black (1992), Plant Cell '
               'Environ 15, 421-429 (hemisurface) or Niinemets et al. (2001) and '
               'Grace (1987), NZ J For Sci 17, 292-294 (projected). Longevity '
               'UNVERIFIED as for c_lf',
        provenance='site measurement'),
    'rh_annual': dict(
        symbol='Rh_annual', role='sampled',
        motivation='290-370 g C m-2 yr-1: measured soil CO2 efflux split between '
                   'root plus rhizosphere respiration and decomposition on a '
                   'girdling experiment at the site',
        source=ILV2009, provenance='site measurement'),
    'theta_som': dict(
        symbol='theta_som', role='sampled',
        motivation='SOM residence time 24-61 yr, solved from the measured soil '
                   'carbon stock 6560 g C m-2 and Rh 290-370 g C m-2 yr-1',
        source=ILV2009 + ', through parameters.som_residence_time_from_stock',
        provenance='site measurement'),
    'f_lab': dict(
        symbol='f_lab', role='simplex',
        motivation='needle litterfall 154 g C m-2 yr-1, split equally with f_fol '
                   'because the measurement constrains their sum, not the split',
        source=ILV2009, provenance='site measurement'),
    'f_fol': dict(
        symbol='f_fol', role='simplex',
        motivation='needle litterfall 154 g C m-2 yr-1, split equally with f_lab',
        source=ILV2009, provenance='site measurement'),
    'f_roo': dict(
        symbol='f_roo', role='simplex',
        motivation='below-ground tree litter about 90 g C m-2 yr-1',
        source=ILV2009, provenance='site measurement'),
    'f_woo': dict(
        symbol='f_woo', role='simplex',
        motivation='above-ground net growth 180-240 plus below-ground 34-69, '
                   'taken as 261 g C m-2 yr-1',
        source=ILV2009, provenance='site measurement'),
    'c_lab_0': dict(
        symbol='C_lab(0)', role='derived',
        motivation='litterfall times f_lab / (f_lab + f_fol): about one year of '
                   'labile accumulation at the 1 January block start',
        source=ILV2009, provenance='site measurement'),
    'c_fol_0': dict(
        symbol='C_fol(0)', role='derived',
        motivation='litterfall divided by c_lf',
        source=ILV2009, provenance='site measurement'),
    'c_roo_0': dict(
        symbol='C_roo(0)', role='derived',
        motivation='below-ground litterfall over theta_roo times 365.25. Measured '
                   'living fine-root carbon 238 g C m-2 is held back as an '
                   'independent check rather than used',
        source=ILV2009 + ' and Table 4', provenance='site measurement'),
    'c_woo_0': dict(
        symbol='C_woo(0)', role='derived',
        motivation='measured tree carbon 6800 g C m-2 less the labile, foliar and '
                   'fine-root pools',
        source=ILV2009, provenance='site measurement'),
    'c_lit_0': dict(
        symbol='C_lit(0)', role='derived',
        motivation='(1 - f_som) rh_ref / theta_lit: the litter stock that respires '
                   'its share of the annual total at the reference temperature',
        source='derived from rh_annual and f_som; ' + ILV2009 + ' for rh_annual',
        provenance='site measurement'),
    'c_som_0': dict(
        symbol='C_som(0)', role='derived',
        motivation='f_som rh_ref / theta_som',
        source='derived from rh_annual and f_som; ' + ILV2009 + ' for rh_annual',
        provenance='site measurement'),
}

missing = sorted(set(PARAMETER_NAMES) - set(PRIOR_PROVENANCE))
print('model parameters with no provenance row:', missing or 'none')

In [ ]:
# Assemble the table and sort weakest provenance first.
PROVENANCE_ORDER = [
    'guess',
    'generic DALEC',
    'published for Scots pine or boreal',
    'site measurement',
]


def describe_prior(name, info):
    """The distribution and its settings, read from the bounds."""
    if info['role'] == 'simplex':
        index = ALLOCATION_WEIGHT_ORDER.index(name)
        alpha = float(DIRICHLET_CONCENTRATION[index])
        return ('Dirichlet component ' + str(index) + ', alpha = '
                + format(alpha, '.4g') + ', scaled by (1 - f_auto)')
    if name in SAMPLED_BOUNDS:
        low, high = SAMPLED_BOUNDS[name]
        return 'Uniform(' + format(low, '.6g') + ', ' + format(high, '.6g') + ')'
    return 'derived, not sampled'


def registry_unit(name):
    entry = PARAMETER_REGISTRY.get(name)
    if entry is not None:
        return entry.unit
    return {'rh_annual': 'g C m-2 yr-1', 'f_som': '1'}.get(name, 'UNVERIFIED')


PRIOR_TABLE = pd.DataFrame([
    {
        'parameter': name,
        'symbol': info['symbol'],
        'unit': registry_unit(name),
        'prior': describe_prior(name, info),
        'motivating value or range': info['motivation'],
        'source': info['source'],
        'provenance': info['provenance'],
        'role': info['role'],
    }
    for name, info in PRIOR_PROVENANCE.items()
])
PRIOR_TABLE['_order'] = PRIOR_TABLE['provenance'].map(PROVENANCE_ORDER.index)
PRIOR_TABLE = (PRIOR_TABLE.sort_values(['_order', 'parameter'])
               .drop(columns='_order')
               .reset_index(drop=True))

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 46)
PRIOR_TABLE[['parameter', 'symbol', 'unit', 'prior', 'provenance', 'role']]

In [ ]:
# The same rows with the motivation and the citation, which are too wide to
# read next to the numbers. Weakest provenance first, as above.
pd.set_option('display.max_colwidth', None)
for _, row in PRIOR_TABLE.iterrows():
    print(row['parameter'], '  [', row['provenance'], '|', row['role'], ']')
    print('    prior      ', row['prior'], '  unit', row['unit'])
    print('    motivation ', row['motivating value or range'])
    print('    source     ', row['source'])
    print()
pd.set_option('display.max_colwidth', 46)

In [ ]:
# Counts by provenance, and the rows that still carry an unverified source.
print(PRIOR_TABLE['provenance'].value_counts().reindex(PROVENANCE_ORDER).to_string())
print()
unverified = PRIOR_TABLE[PRIOR_TABLE['source'].str.contains('UNVERIFIED')]
print('rows with an unverified source:', len(unverified))
for _, row in unverified.iterrows():
    print('  ', row['parameter'].ljust(10), '--', row['provenance'])

### What is weakest here, and where I have made a judgement

**`f_som` is a guess and is labelled as one.** The soil share of heterotrophic respiration is taken as 0.5-0.9 on the reasoning that boreal heterotrophic respiration is dominated by humus and mineral soil rather than fresh litter, but not overwhelmingly so. `src/dalec/parameters.py` records it as a judgement. It is the first row of the table for that reason.

**The needle longevity is unverified and load-bearing twice over.** The 3-5 year figure sets `c_lf` directly and `lma` through it, so if it moves, the foliar pool and the canopy move together and GPP follows. There is a candidate citation but I have not confirmed it against the paper.

**Three provenance assignments are arguable.**

- `ceff` is filed as `generic DALEC` rather than boreal. The REFLEX range is a published prior for the same ACM parameter, not a FI-Hyy measurement, and the paper is recorded as not read in full, so the site type is not established here.
- `theta_lit` is filed as `published for Scots pine or boreal` because Yasso07 is the Finnish soil carbon model and its rates are fitted to a litter-bag dataset -- but they are model parameters, not a FI-Hyy measurement.
- `theta_som` is filed as `site measurement` although no published residence time is used at all: the range is solved from the site's own soil carbon stock and respiration. It is a derivation from two site measurements rather than a measurement.

**`lma` depends on the LAI convention.** Under `hemisurface` it is roughly 116-193 g C m-2 and under `projected` roughly 148-247. Neither convention is adopted in this project; the calibration is run under both and the difference reported. The table shows whichever `CONVENTION` is set to in the config cell.

---

# 5. Prior predictive check

**In one sentence:** draw parameter sets from the priors alone, with no data involved, run the forward model on each, and look at what fluxes come out -- if most of them are physically impossible, the priors are wrong.

It is the one check on a prior that can be made before any fitting, and it has already caught something once in this project: on the published DALEC2 priors, 837 of 1000 draws were unusable, which is the first row of the audit trail in section 3.

The priors examined here are the **current** ones. The failure rate below is therefore not comparable with that 837 -- it is a different prior.

In [ ]:
# The driver block. Real drivers, or idealised substitutes, per section 1(b).
if not CALIBRATION_BLOCK_PATH.exists():
    raise FileNotFoundError(
        'no processed calibration block at ' + str(CALIBRATION_BLOCK_PATH)
        + ' -- run scripts/01_prepare_data.py first'
    )

_full = SiteData.load(CALIBRATION_BLOCK_PATH)
_calendar = pd.DatetimeIndex(_full.time).year.to_numpy()
_keep = (_calendar >= DRIVER_YEARS[0]) & (_calendar <= DRIVER_YEARS[1])
if not _keep.any():
    raise ValueError('no day of the calibration block falls in ' + str(DRIVER_YEARS))

_sliced = {
    field.name: getattr(_full, field.name)[_keep]
    for field in dataclasses.fields(_full)
    if isinstance(getattr(_full, field.name), np.ndarray)
}
_sliced['partitioned'] = {}      # the partitioned GPP and RECO products are
                                 # observation-derived and are dropped here
BLOCK = dataclasses.replace(_full, **_sliced)


def idealised_drivers(block):
    """Smooth sinusoidal substitutes for the real meteorological record."""
    doy = block.doy.astype(float)
    phase = 2.0 * np.pi * (doy - 15.0) / 365.25
    t_air = 3.5 - 11.0 * np.cos(phase)                 # degC, annual cycle
    sw_in = 8.5 - 8.0 * np.cos(phase)                  # MJ m-2 d-1, always positive
    return dataclasses.replace(
        block,
        t_air=t_air,
        t_day=t_air + 2.0,
        t_night=t_air - 2.0,
        t_max=t_air + 4.0,
        t_min=t_air - 4.0,
        sw_in=np.maximum(sw_in, 0.05),
        co2=np.full(block.n_days, 370.0),
    )


if not USE_REAL_DRIVERS:
    BLOCK = idealised_drivers(BLOCK)

# Section 1(a): strip the observed flux series, then assert it is gone. Every
# later cell reads a synthetic series built in section 7 instead.
BLOCK = dataclasses.replace(BLOCK, nee_obs=np.full(BLOCK.n_days, np.nan))
assert_no_observed_flux(BLOCK, 'driver block')

BLOCK_DATES = pd.DatetimeIndex(BLOCK.time)
BLOCK_YEARS = BLOCK_DATES.year.to_numpy()
BLOCK_MONTHS = BLOCK_DATES.month.to_numpy()

print('drivers        ', 'real FI-Hyy record' if USE_REAL_DRIVERS else 'idealised')
print('block          ', BLOCK.n_days, 'days,', BLOCK_DATES[0].date(), 'to',
      BLOCK_DATES[-1].date())
print('observed NEE   ', 'absent (scope guard passed)')
print('t_air range    ', round(float(BLOCK.t_air.min()), 1), 'to',
      round(float(BLOCK.t_air.max()), 1), 'degC')
print('sw_in range    ', round(float(BLOCK.sw_in.min()), 2), 'to',
      round(float(BLOCK.sw_in.max()), 2), 'MJ m-2 d-1')

In [ ]:
# The per-day noise scale and the day mask, per section 1(c) and 1(d).
if USE_REAL_SIGMA:
    _raw = BLOCK.nee_unc
    _usable = np.isfinite(_raw) & (_raw > 0.0)
    if not _usable.any():
        raise ValueError('no usable RANDUNC value in this block')
    SIGMA = np.where(_usable, _raw, np.nanmedian(_raw[_usable]))
else:
    SIGMA = np.full(BLOCK.n_days, float(IDEALISED_SIGMA))

if USE_REAL_DAY_MASK:
    DAY_MASK = BLOCK.nee_mask.astype(bool) & np.isfinite(SIGMA) & (SIGMA > 0.0)
else:
    DAY_MASK = np.ones(BLOCK.n_days, dtype=bool)

if not DAY_MASK.any():
    raise ValueError('no day passes the mask, so there is nothing to fit')

print('sigma source   ', 'real RANDUNC' if USE_REAL_SIGMA else 'constant '
      + str(IDEALISED_SIGMA))
print('sigma range    ', round(float(SIGMA.min()), 3), 'to',
      round(float(SIGMA.max()), 3), 'g C m-2 d-1')
print('sigma median   ', round(float(np.median(SIGMA)), 3))
print('day mask       ', 'real QC mask' if USE_REAL_DAY_MASK else 'every day')
print('days in fit    ', int(DAY_MASK.sum()), 'of', BLOCK.n_days)

In [ ]:
# The true parameter set: one draw from the prior in use, at its own seed.
#
# Drawing the truth from the prior is deliberate. It makes the prior correct by
# construction, which is what isolates the likelihood as the only thing being
# tested in Part B. It also means Part B says nothing about prior
# misspecification, which is Part A's job.
ACM = acm_from_config(CONFIG)

try:
    _truth_list, TRUTH_FRAME = sample_reparameterised_parameters(
        1, rng=np.random.default_rng(SEED_TRUTH), t_air=BLOCK.t_air
    )
except ValueError as exc:
    raise SystemExit(
        'the truth draw was rejected by the prior construction: ' + str(exc)
        + ' -- change SEED_TRUTH in the config cell and re-run'
    ) from exc

TRUTH = _truth_list[0]
TRUE_VALUES = {name: float(getattr(TRUTH, name)) for name in PARAMETER_NAMES}

print('true parameter set drawn with seed', SEED_TRUTH)
print()
_show = ['f_auto', 'c_lf', 'lma', 'ceff', 'temperature_exponent',
         'theta_lit', 'theta_som', 'c_fol_0', 'c_som_0']
for name in _show:
    print('  ' + name.ljust(22), format(TRUE_VALUES[name], '.6g'))
print('  ...', len(TRUE_VALUES) - len(_show), 'further fields')

In [ ]:
# Draw N parameter sets from the prior and run each one forward.
#
# Stored per draw: daily NEE, daily GPP, the minimum each pool reaches, the
# largest growth factor any pool undergoes, and the maximum LAI. Full pool
# trajectories are not kept -- six times the size of NEE, and nothing below
# needs them.
#
# LAI = C_fol / lma. In words: leaf area follows from the foliar carbon a draw
# grows and the leaf mass per unit area it was given.
_rng = np.random.default_rng(SEED_PRIOR_DRAWS)
PRIOR_PARAMS, PRIOR_FRAME = sample_reparameterised_parameters(
    N_PRIOR_DRAWS, rng=_rng, t_air=BLOCK.t_air
)

n_days = BLOCK.n_days
n_draws = len(PRIOR_PARAMS)
prior_run = {
    'nee': np.full((n_draws, n_days), np.nan),
    'gpp': np.full((n_draws, n_days), np.nan),
    'pool_min': np.full((n_draws, len(POOL_NAMES)), np.nan),
    'pool_growth': np.full(n_draws, np.nan),
    'lai_max': np.full(n_draws, np.nan),
    'ran': np.zeros(n_draws, dtype=bool),
    'error': [''] * n_draws,
}

_started = time.perf_counter()
for index, params in enumerate(PRIOR_PARAMS):
    try:
        output = run_dalec2(params, BLOCK, gpp_fn=ACM)
    except Exception as exc:                    # recorded, not hidden
        prior_run['error'][index] = type(exc).__name__ + ': ' + str(exc)[:140]
        continue
    prior_run['ran'][index] = True
    prior_run['nee'][index] = output.nee
    prior_run['gpp'][index] = output.gpp
    prior_run['pool_min'][index] = output.pools.min(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        growth = np.where(output.pools[0] > 0.0,
                          output.pools.max(axis=0) / output.pools[0], np.inf)
    prior_run['pool_growth'][index] = float(np.max(growth))
    prior_run['lai_max'][index] = float(
        np.max(leaf_area_index(output.pool('c_fol'), params.lma)))
    if (index + 1) % PROGRESS_EVERY == 0:
        print('  ', index + 1, 'of', n_draws, 'draws,',
              round(time.perf_counter() - _started, 1), 's', flush=True)

prior_run['finite'] = np.isfinite(prior_run['nee']).all(axis=1)
prior_run['evaluable'] = prior_run['ran'] & prior_run['finite']

print()
print('draws          ', n_draws)
print('integrated     ', int(prior_run['ran'].sum()))
print('finite output  ', int(prior_run['evaluable'].sum()))
print('elapsed        ', round(time.perf_counter() - _started, 1), 's')

**Figure 1.** Prior predictive net ecosystem exchange across one year. The bands show where the prior draws fall on each day: the darker band holds the central half of the draws, the lighter band the central 90 per cent, and the line is the median. Negative NEE is net uptake by the forest, following the sign convention of the flux data.

The only reference lines are physically motivated: zero, which separates a carbon source from a carbon sink, and the screening limit this project uses for an implausible daily flux. **No observed flux is drawn.**

In [ ]:
FAN_PERCENTILES = (5.0, 25.0, 50.0, 75.0, 95.0)
FAN_YLIM = (-14.0, 14.0)

fan_index = np.flatnonzero(BLOCK_YEARS == FAN_YEAR)
if fan_index.size == 0:
    raise ValueError('FAN_YEAR ' + str(FAN_YEAR) + ' is not inside DRIVER_YEARS '
                     + str(DRIVER_YEARS))

usable = prior_run['ran'] & np.isfinite(prior_run['nee'][:, fan_index]).all(axis=1)
fan_nee = prior_run['nee'][usable][:, fan_index]
bands = np.percentile(fan_nee, FAN_PERCENTILES, axis=0)
dates = BLOCK_DATES[fan_index]

figure, axis = plt.subplots(figsize=FIGSIZE_WIDE)
axis.fill_between(dates, bands[0], bands[4], color=COLOUR['prior'], alpha=0.22,
                  label='prior draws, central 90 per cent')
axis.fill_between(dates, bands[1], bands[3], color=COLOUR['prior'], alpha=0.42,
                  label='prior draws, central 50 per cent')
axis.plot(dates, bands[2], color=COLOUR['prior'], lw=1.6, label='prior median')
axis.axhline(0.0, color=COLOUR['neutral'], lw=1.0,
             label='zero: source above, sink below')
for sign in (-1.0, 1.0):
    limit = sign * FAILURE_NEE_LIMIT
    if FAN_YLIM[0] <= limit <= FAN_YLIM[1]:
        axis.axhline(limit, color=COLOUR['accent'], lw=0.9, ls='--')
axis.set_ylim(*FAN_YLIM)
axis.set_ylabel('NEE, g C m$^{-2}$ d$^{-1}$')
axis.set_xlabel(str(FAN_YEAR))
axis.set_title('Figure 1. Prior predictive NEE, ' + str(fan_nee.shape[0])
               + ' draws, ' + CONVENTION + ' convention')
axis.legend(loc='upper left', ncol=2)
plt.show()

print('draws in the fan          ', int(usable.sum()), 'of', n_draws)
print('draws outside the y-limits',
      int((np.abs(fan_nee).max(axis=1) > FAN_YLIM[1]).sum()))
print('screening limit drawn at  ', FAILURE_NEE_LIMIT,
      'g C m-2 d-1 (dalec.diagnostics.FAILURE_NEE_LIMIT)')

### 5c. Physical plausibility rules

Each rule is a separate named function returning one boolean per draw, where `True` means that draw **fails** that rule. They are independent: a draw can fail several. Each carries its justification in the table, and the justification is of one of three kinds -- definitional, sourced from a site measurement, or a stated assumption.

One rule is a stated assumption and needs flagging now rather than in the footnotes. **The annual NEE window has no source.** The natural bound would be the site's own measured annual NEE, but that is an observed flux and section 1(a) excludes it from this notebook; a published range for boreal pine stands would be an observed flux too. The window is therefore set wide enough to screen out impossible ecosystems rather than atypical years, and it is recorded as an assumption. Whether that is the right call is the first open question in section 12.

In [ ]:
# Annual NEE per draw, over calendar years long enough to sum.
complete_years = [
    int(year) for year in sorted(set(BLOCK_YEARS.tolist()))
    if int((BLOCK_YEARS == year).sum()) >= MIN_DAYS_FOR_ANNUAL
]
if complete_years:
    prior_run['annual_nee'] = np.column_stack([
        prior_run['nee'][:, BLOCK_YEARS == year].sum(axis=1)
        for year in complete_years
    ])
else:
    prior_run['annual_nee'] = np.full((n_draws, 1), np.nan)

print('complete calendar years in the block:', complete_years or 'none')
if not complete_years:
    print('the annual NEE rule cannot be evaluated; it will report as not evaluable')

In [ ]:
# The rules. True means the draw fails.

def rule_integration_raised(run):
    return ~run['ran']


def rule_non_finite_flux(run):
    return run['ran'] & ~run['finite']


def rule_gpp_below_zero(run):
    return (np.nanmin(run['gpp'], axis=1) < GPP_FLOOR_G_C_M2_D) & run['evaluable']


def rule_pool_negative(run):
    return (run['pool_min'].min(axis=1) < 0.0) & run['evaluable']


def rule_lai_above_ceiling(run):
    return (run['lai_max'] > LAI_CEILING) & run['evaluable']


def rule_pool_growth_above_limit(run):
    return (run['pool_growth'] > FAILURE_GROWTH_FACTOR) & run['evaluable']


def rule_annual_nee_outside_range(run):
    low, high = ANNUAL_NEE_RANGE_G_C_M2_YR
    outside = (run['annual_nee'] < low) | (run['annual_nee'] > high)
    return outside.any(axis=1) & run['evaluable']


RULES = {
    'integration_raised': (
        rule_integration_raised,
        'Numerical. The forward model raised rather than returning a series.'),
    'non_finite_flux': (
        rule_non_finite_flux,
        'Numerical. A NaN or infinity anywhere in the daily NEE series.'),
    'gpp_below_zero': (
        rule_gpp_below_zero,
        'Definitional. GPP is gross uptake; a negative value is not a physical '
        'state. No citation needed.'),
    'pool_negative': (
        rule_pool_negative,
        'Definitional. A carbon stock cannot be negative.'),
    'lai_above_ceiling': (
        rule_lai_above_ceiling,
        'Site measurement. Kolari (2010), Dissertationes Forestales 99, gives the '
        'all-sided seasonal maximum LAI at SMEAR II as 6.5-8.0; converted to the '
        'chosen basis by dalec.parameters.site_lai. LAI is a canopy measurement, '
        'not an observed flux, so it is admissible under section 1. The ceiling '
        'is LAI_CEILING_MULTIPLE times that maximum, and the multiple is a '
        'stated tolerance, not published.'),
    'pool_growth_above_limit': (
        rule_pool_growth_above_limit,
        'Screen recorded in this project. dalec.diagnostics.FAILURE_GROWTH_FACTOR '
        '= ' + format(FAILURE_GROWTH_FACTOR, 'g') + 'x growth in any pool over the '
        'block, used by the prior diagnostics of section 3.'),
    'annual_nee_outside_range': (
        rule_annual_nee_outside_range,
        'STATED ASSUMPTION, no source. # TODO. The site measured value is excluded '
        'by section 1(a); see the markdown above and question 1 of section 12.'),
}

RULE_SCOPE = {
    'integration_raised': np.ones(n_draws, dtype=bool),
    'non_finite_flux': prior_run['ran'],
    'gpp_below_zero': prior_run['evaluable'],
    'pool_negative': prior_run['evaluable'],
    'lai_above_ceiling': prior_run['evaluable'],
    'pool_growth_above_limit': prior_run['evaluable'],
    'annual_nee_outside_range': (
        prior_run['evaluable'] if complete_years else np.zeros(n_draws, dtype=bool)),
}

RULE_FAILURES = {name: rule(prior_run) for name, (rule, _) in RULES.items()}
ANY_FAILURE = np.zeros(n_draws, dtype=bool)
for flags in RULE_FAILURES.values():
    ANY_FAILURE |= flags

print('rules defined:', len(RULES))

In [ ]:
# Table of rule, failing draws, percentage, and the justification.
rows = []
for name, (_, justification) in RULES.items():
    flags = RULE_FAILURES[name]
    scope = RULE_SCOPE[name]
    rows.append({
        'rule': name,
        'failing draws': int(flags.sum()),
        'per cent of all draws': round(100.0 * flags.sum() / n_draws, 2),
        'not evaluable': int((~scope).sum()),
        'justification': justification,
    })
rows.append({
    'rule': 'ANY RULE',
    'failing draws': int(ANY_FAILURE.sum()),
    'per cent of all draws': round(100.0 * ANY_FAILURE.sum() / n_draws, 2),
    'not evaluable': 0,
    'justification': 'union of the rules above',
})
RULE_TABLE = pd.DataFrame(rows)

print('draws', n_draws, '| block', BLOCK.n_days, 'days | LAI ceiling',
      round(LAI_CEILING, 2), '| annual NEE window', ANNUAL_NEE_RANGE_G_C_M2_YR)
print()
RULE_TABLE[['rule', 'failing draws', 'per cent of all draws', 'not evaluable']]

In [ ]:
# The justifications in full.
pd.set_option('display.max_colwidth', None)
for _, row in RULE_TABLE.iterrows():
    print(row['rule'], ' --', row['failing draws'], 'failing')
    print('   ', row['justification'])
    print()
pd.set_option('display.max_colwidth', 46)

### 5d. Which prior drives the failures

For each rule that some draws fail and some pass, the cell below compares the two groups one parameter at a time, using the two-sample **Kolmogorov-Smirnov statistic**: the largest vertical gap between the cumulative distribution of that parameter among the failing draws and among the passing ones. It runs from 0 to 1. *In words: how far apart the failing and passing draws sit on that one parameter's axis, ignoring every other parameter.*

**What it cannot do.** The statistic is univariate. A failure caused by a *combination* of two parameters, neither unusual on its own, shows weakly on both. That is not hypothetical here: the v1 to v2 revision in section 3 exists precisely because `theta_som` and `c_som_0` were individually meaningless and jointly impossible.

In [ ]:
from scipy.stats import ks_2samp

BREAKDOWN_PARAMETERS = [c for c in PRIOR_FRAME.columns if c != 'draw']


def breakdown_for_rule(name, flags, scope):
    """Per-parameter separation between the failing and the passing draws."""
    failing, passing = flags & scope, (~flags) & scope
    if failing.sum() == 0 or passing.sum() == 0:
        return None
    rows = []
    for column in BREAKDOWN_PARAMETERS:
        values = PRIOR_FRAME[column].to_numpy()
        test = ks_2samp(values[failing], values[passing])
        rows.append({
            'rule': name,
            'parameter': column,
            'ks': float(test.statistic),
            'p': float(test.pvalue),
            'median_fail': float(np.median(values[failing])),
            'median_pass': float(np.median(values[passing])),
        })
    return pd.DataFrame(rows).sort_values('ks', ascending=False).reset_index(drop=True)


BREAKDOWNS = {}
for name, flags in list(RULE_FAILURES.items()) + [('ANY RULE', ANY_FAILURE)]:
    scope = RULE_SCOPE.get(name, np.ones(n_draws, dtype=bool))
    result = breakdown_for_rule(name, flags, scope)
    if result is not None:
        BREAKDOWNS[name] = result

if BREAKDOWNS:
    print('rules with both failing and passing draws:', list(BREAKDOWNS))
    print()
    for name, frame in BREAKDOWNS.items():
        print(name, '-- top 6 parameters by separation')
        print(frame.head(6).to_string(index=False))
        print()
else:
    print('no rule produced both failing and passing draws, so there is nothing '
          'to attribute')

**Figure 2.** Separation between failing and passing prior draws, per parameter and per rule. A tall bar means the failing draws sit noticeably elsewhere on that parameter's axis; a short bar means that parameter looks the same in both groups. A short bar is not evidence of innocence -- see the caveat above about joint causes.

In [ ]:
if BREAKDOWNS:
    wide = (pd.concat(BREAKDOWNS.values())
            .pivot(index='parameter', columns='rule', values='ks'))
    wide = wide.loc[wide.max(axis=1).sort_values(ascending=False).index]
    figure, axis = plt.subplots(figsize=(1.5 * wide.shape[1] + 4,
                                         0.30 * len(wide) + 2))
    image = axis.imshow(wide.to_numpy(), aspect='auto', cmap='magma_r',
                        vmin=0.0, vmax=1.0)
    axis.set_xticks(range(wide.shape[1]))
    axis.set_xticklabels(wide.columns, rotation=30, ha='right')
    axis.set_yticks(range(len(wide)))
    axis.set_yticklabels(wide.index)
    axis.grid(False)
    axis.set_title('Figure 2. Which prior separates the failing draws')
    figure.colorbar(image, ax=axis, label='KS statistic')
    plt.show()
else:
    print('Figure 2 not drawn: no rule produced both failing and passing draws.')

---

# 6. What to do if the prior predictive check fails

Options with their costs, not a recommendation. Which applies depends on what section 5 shows, and more than one can apply at once.

| Option | What it fixes | What it costs |
|---|---|---|
| **Tighten bounds using a site measurement** | Removes prior mass that the site's own data says is impossible. Directly addressed the canopy failure in v4. | Only available where a measurement exists, and it moves the parameter from `generic DALEC` to `site measurement` only if the measurement is genuinely independent of the fit. A bound tightened because the fit preferred it is not a prior. |
| **Reparameterise so physically linked parameters cannot drift apart** | Puts the prior on the quantity that is actually constrained. The Dirichlet on the allocation simplex is the example: the fractions must sum to one, so they are drawn as proportions rather than independently. | The sampled quantities are no longer the ones the literature tabulates, so every published range has to be translated, and the translation has to be recorded. The Dirichlet's support is also a superset of the published marginals, which has to be stated rather than hidden. |
| **Change the distribution family** | A uniform prior on a rate spanning four orders of magnitude places 90 per cent of its mass in the top decade, which is a strong claim rather than ignorance. A log-uniform prior says something different and often more defensible. | Bounded uniform priors over published ranges are a locked design decision in this project, so this is a change of decision, not a tweak. It also changes what "the published range" means, which has to be defended. |
| **Put the prior on a derived quantity with a measurable meaning** | Lets the prior be stated where a measurement exists. Sampling the annual respiration total and the soil share, then deriving the two stocks, is exactly this: the flux is measurable, the stocks are not. | The derived quantity has to be a deterministic function of the sampled ones, and its implied prior is then whatever that function produces -- which needs checking, not assuming. |
| **Rescale units** | Removes numerical stiffness when parameters differ by many orders of magnitude, which can make the sampler's step size adaptation struggle. | Cosmetic with respect to the physics, so it fixes sampling difficulty, not implausibility. If the fluxes are impossible, rescaling will not make them possible. |
| **Widen rather than narrow** | Correct when the failures come from a *missing process* rather than a bad prior: the parameters are being asked to compensate for something the model does not represent, and narrowing them forces the compensation somewhere worse. | Admits that the model is wrong in a way the calibration cannot repair, and widens the posterior accordingly. Honest, and unpopular. |

### The option that is closed to me, and what the alternative costs

**Hard ecological constraints applied as an accept/reject step are off the table.** In the DALEC literature these are "ecological dynamical constraints": rules such as *the foliar pool must not decline by more than a given factor over the record*, applied by discarding any parameter set that violates them.

The reason they are closed is the sampler. NUTS moves by following the gradient of the log posterior, and a rejection rule makes that surface discontinuous: inside the allowed region the gradient says nothing about where the boundary is, and at the boundary the log posterior drops to minus infinity. The sampler cannot differentiate through that. In practice it stalls against the wall and produces divergent transitions rather than exploring. This is the same mechanism as the v3 row in section 3, where an allocation fraction taken by subtraction could go negative.

**The soft-penalty alternative** replaces the hard rule with a smooth penalty added to the log posterior -- a term that grows as the constraint is approached rather than a cliff at it. It restores differentiability, and it costs three things:

1. The penalty's shape and strength are free choices, and the posterior depends on them. They would need defending as my own modelling choice, since no published DALEC study states them.
2. It is no longer the published constraint. Reporting it as "EDCs applied" would be wrong; it would be a different model.
3. It changes what the prior *is*. A penalty on a dynamic outcome of the forward model is a prior on trajectories, not on parameters, and its implied marginal prior on any single parameter is not knowable in closed form.

This project's choice is the third route: **reparameterise so that the constraint holds by construction**, which is what the simplex and the derived pools do. That has no tuning constant and no cliff, but it only works for constraints that can be built into the parameterisation -- which not all of them can.

---
---

# PART B: THE LIKELIHOOD

---

# 7. The likelihood in use

For each day $d$ that enters the fit:

$$y_d \;=\; m_d(\theta) \;+\; \varepsilon_d, \qquad \varepsilon_d \sim \mathcal{N}\!\left(0,\, \sigma_d^2\right)$$

*In words: the observed flux on a day is what the model predicts for that day plus a random error, and that error is drawn from a bell curve whose width is the uncertainty the dataset states for that day.*

Written as a log-likelihood, which is what the sampler actually uses:

$$\log p(y \mid \theta) \;=\; -\frac{1}{2}\sum_{d \in D} \left(\frac{y_d - m_d(\theta)}{\sigma_d}\right)^{2} \;-\; \sum_{d \in D}\log \sigma_d \;-\; \frac{n}{2}\log 2\pi$$

*In words: add up the squared mismatch on every included day, each one divided by that day's uncertainty, and the parameter set with the smallest total is the one the data prefers.*

$\sigma_d$ is the `NEE_VUT_REF_RANDUNC` value for that day -- the FLUXNET2015 estimate of the random uncertainty of the flux measurement (Pastorello et al. 2020, *Scientific Data* 7, 225). It is **not** estimated during the fit; it is supplied.

$D$ is the set of days that pass the quality mask. Masked days are excluded by *indexing*, not by multiplying their term by zero: a zero weight still evaluates the term, still records it for the gradient, and lets a non-finite value reach the sum as $0 \times \infty$.

### The four assumptions this carries

1. **The error shape is Gaussian.** Errors of a given size are as likely as a bell curve says they are, and large errors are very unlikely.
2. **The per-day uncertainty is the right scale.** $\sigma_d$ is the standard deviation of the thing that actually separates $y_d$ from $m_d(\theta)$.
3. **Errors are independent from one day to the next.** Knowing that yesterday's model was too high tells you nothing about today.
4. **Errors have zero mean.** The model is not systematically above or below the observations.

Section 8 takes these one at a time. One point applies to all four and is worth making before any of them: $\varepsilon_d$ is written as if it were *measurement* error, but it is really the total discrepancy between the model and the observation. That includes everything the model does not represent. Assumptions 3 and 4 are plausible for measurement noise and much less plausible for model error, and the likelihood as written cannot tell them apart.

---

## 8. The four assumptions, one at a time

Each assumption gets the same three parts: what it says in plain English, why it might be wrong here, and a synthetic demonstration of what it costs if it is wrong.

The demonstrations share one cheap device. Rather than re-running the sampler four times, they hold every parameter at its true value, vary one, and look at the log-likelihood along that line. That is a **profile log-likelihood**: *a slice through the likelihood surface in one direction, with everything else pinned at the truth*. It is not a posterior, and the width of its peak is not a posterior standard deviation. What it is good for is showing how the likelihood's preferred value and its apparent sharpness respond when an assumption is broken -- and it costs one forward-model run per grid point, once, reused by every demonstration below.

In [ ]:
# The scan: vary one parameter, hold the rest at truth, store the predicted
# NEE series for each grid value. Every section 8 demonstration reuses these.
if SCAN_PARAMETER in ALLOCATION_PARAMETERS:
    raise ValueError(
        'SCAN_PARAMETER cannot be an allocation fraction: changing one alone '
        'breaks the closure f_auto + f_lab + f_fol + f_roo + f_woo = 1'
    )

SCAN_LOW, SCAN_HIGH = SAMPLED_BOUNDS[SCAN_PARAMETER]
TRUE_SCAN_VALUE = TRUE_VALUES[SCAN_PARAMETER]
half_span = 0.5 * SCAN_SPAN * (SCAN_HIGH - SCAN_LOW)
SCAN_VALUES = np.linspace(max(SCAN_LOW, TRUE_SCAN_VALUE - half_span),
                          min(SCAN_HIGH, TRUE_SCAN_VALUE + half_span),
                          SCAN_POINTS)

SCAN_PREDICTIONS = np.full((SCAN_POINTS, n_days), np.nan)
_started = time.perf_counter()
for index, value in enumerate(SCAN_VALUES):
    params = dataclasses.replace(TRUTH, **{SCAN_PARAMETER: float(value)})
    SCAN_PREDICTIONS[index] = run_dalec2(params, BLOCK, gpp_fn=ACM).nee

TRUE_NEE = run_dalec2(TRUTH, BLOCK, gpp_fn=ACM).nee

print('scan parameter ', SCAN_PARAMETER, '(', PARAMETER_REGISTRY[SCAN_PARAMETER].unit, ')')
print('prior range    ', format(SCAN_LOW, '.4g'), 'to',
      format(SCAN_HIGH, '.4g'))
print('scanned        ', format(SCAN_VALUES[0], '.4g'), 'to',
      format(SCAN_VALUES[-1], '.4g'), 'in', SCAN_POINTS, 'steps')
print('true value     ', format(TRUE_SCAN_VALUE, '.6g'))
print('forward runs   ', SCAN_POINTS + 1, 'in',
      round(time.perf_counter() - _started, 1), 's')

In [ ]:
# Log-likelihood helpers, and a quadratic estimate of how sharp a profile is.
from scipy.stats import t as student_t


def gaussian_profile(observations, sigma, mask, predictions):
    """Gaussian log-likelihood at every scan point. Shape (SCAN_POINTS,)."""
    residual = (observations[mask] - predictions[:, mask]) / sigma[mask]
    constant = np.log(sigma[mask]).sum() + 0.5 * mask.sum() * np.log(2.0 * np.pi)
    return -0.5 * (residual ** 2).sum(axis=1) - constant


def student_profile(observations, sigma, mask, predictions, nu):
    """Student-t log-likelihood at every scan point, same scale sigma."""
    return student_t.logpdf(
        observations[mask], df=nu, loc=predictions[:, mask], scale=sigma[mask]
    ).sum(axis=1)


def profile_peak_and_width(values, loglik, window=9):
    """Fit a parabola near the maximum; return (peak location, implied width).

    For a quadratic log-likelihood, log L = c - (x - x0)^2 / (2 s^2), so the
    fitted curvature gives s. In words: how sharply the likelihood falls away
    from its best value, expressed as a standard deviation in the units of the
    parameter. This is a one-parameter Laplace approximation with everything
    else held at truth -- it is not a posterior standard deviation.
    """
    finite = np.isfinite(loglik)
    if finite.sum() < 3:
        return np.nan, np.nan
    peak_index = int(np.nanargmax(np.where(finite, loglik, -np.inf)))
    low = max(0, peak_index - window // 2)
    high = min(len(values), low + window)
    low = max(0, high - window)
    x, y = values[low:high], loglik[low:high]
    keep = np.isfinite(y)
    if keep.sum() < 3:
        return np.nan, np.nan
    quadratic, linear, _ = np.polyfit(x[keep], y[keep], 2)
    if quadratic >= 0.0:
        return float(values[peak_index]), np.nan
    return float(-linear / (2.0 * quadratic)), float(1.0 / np.sqrt(-2.0 * quadratic))


print('profile helpers ready')

In [ ]:
# The baseline synthetic dataset: truth plus Gaussian noise at the per-day
# scale of section 1(c). This is the dataset the likelihood is correct for.
_noise_rng = np.random.default_rng(SEED_NOISE)
OBS_GAUSSIAN = TRUE_NEE + _noise_rng.normal(0.0, SIGMA)

BASELINE_PEAK, CLAIMED_WIDTH = profile_peak_and_width(
    SCAN_VALUES, gaussian_profile(OBS_GAUSSIAN, SIGMA, DAY_MASK, SCAN_PREDICTIONS))

print('baseline synthetic dataset built')
print('  days contributing    ', int(DAY_MASK.sum()))
print('  true ' + SCAN_PARAMETER.ljust(16), format(TRUE_SCAN_VALUE, '.6g'))
print('  profile peak         ', format(BASELINE_PEAK, '.6g'))
print('  implied width (1 sd) ', format(CLAIMED_WIDTH, '.4g'))

### 8.1 Assumption 1: the error shape is Gaussian

**What it says.** Errors of a given size occur as often as a bell curve says. In particular, an error four times the stated uncertainty is essentially ruled out: under a Gaussian it happens on about 6 days in 100,000.

**Why it might be wrong here.** Eddy-covariance flux errors are widely reported to be heavier-tailed than Gaussian -- closer to a double-exponential shape, meaning occasional large excursions are far more common than a bell curve allows. The usual mechanism is that the flux is a turbulent covariance measured over a finite window, and occasional non-stationary conditions produce large, real, one-off departures.

> **Candidate references, UNVERIFIED -- check before citing.**
> Hollinger, D. Y. and Richardson, A. D. (2005), *Tree Physiology* 25, 873-885; and Richardson, A. D. et al. (2006), *Agricultural and Forest Meteorology* 136, 1-18. Both are commonly cited for a double-exponential error distribution in tower flux data. **# TODO:** obtain both, confirm the distributional claim and the table it appears in, and confirm whether it is stated at half-hourly or at daily aggregation -- the aggregation matters, because averaging pushes a sum towards a Gaussian.

**What is demonstrated below.** A second synthetic dataset is built with heavy-tailed noise, scaled so that its standard deviation is the same as the Gaussian case. So the two differ in *shape*, not in overall noise level. The profile log-likelihood is then computed three ways, and the peak location and apparent width compared against the known truth.

In [ ]:
# Heavy-tailed noise with the same standard deviation as the Gaussian case.
#
# A Student-t with nu degrees of freedom and scale s has variance
# s^2 * nu / (nu - 2), so the scale is shrunk by sqrt((nu - 2) / nu) to match.
# In words: the two noise series are equally large on average, and differ only
# in how often they produce an extreme day.
if HEAVY_TAIL_NU <= 2.0:
    raise ValueError('HEAVY_TAIL_NU must exceed 2 for the variance to exist')

_scale = SIGMA * np.sqrt((HEAVY_TAIL_NU - 2.0) / HEAVY_TAIL_NU)
_heavy_rng = np.random.default_rng(SEED_NOISE + 1)
HEAVY_NOISE = _scale * student_t.rvs(df=HEAVY_TAIL_NU, size=n_days,
                                     random_state=_heavy_rng)
OBS_HEAVY = TRUE_NEE + HEAVY_NOISE

_z = HEAVY_NOISE[DAY_MASK] / SIGMA[DAY_MASK]
_zg = (OBS_GAUSSIAN - TRUE_NEE)[DAY_MASK] / SIGMA[DAY_MASK]
print('noise in units of the stated sigma, on days that enter the fit:')
print('               Gaussian    heavy-tailed')
print('  sd          ', format(_zg.std(ddof=1), '.3f').rjust(8),
      format(_z.std(ddof=1), '.3f').rjust(14))
print('  largest |z| ', format(np.abs(_zg).max(), '.3f').rjust(8),
      format(np.abs(_z).max(), '.3f').rjust(14))
print('  days |z| > 3', str(int((np.abs(_zg) > 3).sum())).rjust(8),
      str(int((np.abs(_z) > 3).sum())).rjust(14))

**Figure 3.** Profile log-likelihood along one parameter, with everything else held at its true value. Each curve is shifted so its own maximum is zero, so the comparison is of *shape and location*, not of height. The vertical line is the true value, which is known because the data is synthetic.

Three curves: the Gaussian likelihood on Gaussian data (the case it is correct for), the Gaussian likelihood on heavy-tailed data (the misspecified case), and a heavier-tailed likelihood on the same heavy-tailed data.

In [ ]:
profiles = {
    'Gaussian likelihood, Gaussian data': (
        gaussian_profile(OBS_GAUSSIAN, SIGMA, DAY_MASK, SCAN_PREDICTIONS),
        COLOUR['gaussian'], '-'),
    'Gaussian likelihood, heavy-tailed data': (
        gaussian_profile(OBS_HEAVY, SIGMA, DAY_MASK, SCAN_PREDICTIONS),
        COLOUR['heavy'], '-'),
    'Student-t likelihood, heavy-tailed data': (
        student_profile(OBS_HEAVY, SIGMA, DAY_MASK, SCAN_PREDICTIONS, HEAVY_TAIL_NU),
        COLOUR['accent'], '--'),
}

figure, axis = plt.subplots(figsize=FIGSIZE_WIDE)
summary = []
for label, (loglik, colour, style) in profiles.items():
    axis.plot(SCAN_VALUES, loglik - np.nanmax(loglik), color=colour, ls=style,
              label=label)
    peak, width = profile_peak_and_width(SCAN_VALUES, loglik)
    summary.append({'profile': label, 'peak': peak,
                    'peak minus truth': peak - TRUE_SCAN_VALUE, 'implied width': width})
axis.axvline(TRUE_SCAN_VALUE, color=COLOUR['truth'], lw=2.0, label='true value')
axis.set_xlabel(SCAN_PARAMETER + ', ' + PARAMETER_REGISTRY[SCAN_PARAMETER].unit)
axis.set_ylabel('log-likelihood minus its own maximum')
axis.set_ylim(-25.0, 2.0)
axis.set_title('Figure 3. Does the error shape matter?')
axis.legend(loc='lower center')
plt.show()

pd.DataFrame(summary).set_index('profile')

**How to read the table and the figure.** Three readings are possible and they are not exclusive:

- The peak of the misspecified curve sits away from the true value, by more or less than the curve's own implied width. That is bias.
- The misspecified curve is narrower or wider than the correctly specified one. A narrower curve on worse assumptions means the fit is more confident and less correct.
- The three curves are indistinguishable at this noise level and this number of days, in which case the assumption is not costing anything detectable here -- which is a result about this block, not about the assumption.

### 8.2 Assumption 2: the per-day uncertainty is the right scale

**What it says.** The number in the `RANDUNC` column is the standard deviation of whatever separates the observation from the model on that day.

**Why it might be wrong here.** `RANDUNC` estimates the *random measurement* uncertainty of the flux. The discrepancy the likelihood actually sees is measurement error plus model error -- everything DALEC2 does not represent. At this site that list is not short: there is no explicit hydrology, the site has four to six months of soil frost against the three months the model's design assumed, and the modelled growing season is currently about 41 days longer than the record suggests. None of that is in `RANDUNC`.

If the true discrepancy is larger than $\sigma_d$, the likelihood is overconfident: it treats a model failure as an improbable measurement, and pays a very large penalty for it.

> **Source for what RANDUNC is:** Pastorello, G. et al. (2020), *Scientific Data* 7, 225, the FLUXNET2015 dataset paper. **# TODO:** cite the specific section describing the random uncertainty estimate rather than the paper as a whole.

**What is demonstrated below.** Two different ways of getting $\sigma$ wrong, which have different consequences:

1. A **uniform** misstatement: every day's $\sigma$ multiplied by the same factor.
2. A **structured** misstatement: $\sigma$ inflated in summer only, which is what a seasonal model error would look like if it were absorbed into the noise.

In [ ]:
# Uniform and structured misstatements of sigma, on the baseline dataset.
summer = np.isin(BLOCK_MONTHS, SUMMER_MONTHS)

rows = []
for factor in SIGMA_SCALES:
    peak, width = profile_peak_and_width(
        SCAN_VALUES,
        gaussian_profile(OBS_GAUSSIAN, SIGMA * factor, DAY_MASK, SCAN_PREDICTIONS))
    rows.append({'sigma used': 'uniform x ' + format(factor, '.2f'),
                 'peak': peak, 'peak minus truth': peak - TRUE_SCAN_VALUE,
                 'implied width': width})

_structured = np.where(summer, SIGMA * SUMMER_SIGMA_INFLATION, SIGMA)
peak, width = profile_peak_and_width(
    SCAN_VALUES, gaussian_profile(OBS_GAUSSIAN, _structured, DAY_MASK,
                                  SCAN_PREDICTIONS))
rows.append({'sigma used': 'summer x ' + format(SUMMER_SIGMA_INFLATION, '.2f')
             + ', elsewhere x 1', 'peak': peak,
             'peak minus truth': peak - TRUE_SCAN_VALUE, 'implied width': width})

SIGMA_SCALE_TABLE = pd.DataFrame(rows).set_index('sigma used')
print('true', SCAN_PARAMETER, '=', format(TRUE_SCAN_VALUE, '.6g'))
print('summer days in the block:', int((summer & DAY_MASK).sum()),
      'of', int(DAY_MASK.sum()), 'contributing days')
print()
SIGMA_SCALE_TABLE

**Figure 4.** What a misstated uncertainty does to the width of the likelihood. The horizontal axis is the factor every day's $\sigma$ was multiplied by; the vertical axis is the implied width of the profile peak. The dashed line marks the correct factor of 1.

There is an algebraic point to check against the figure: multiplying every $\sigma$ by a constant $k$ divides the whole sum of squared residuals by $k^2$, which changes the *curvature* of the profile but not the location of its maximum. So a uniform error in $\sigma$ misstates the uncertainty without biasing the estimate. A *structured* error does both, which is why the structured case is shown as a separate marker.

In [ ]:
figure, axis = plt.subplots(figsize=(7.0, 4.4))
factors = np.array(SIGMA_SCALES, dtype=float)
widths = SIGMA_SCALE_TABLE['implied width'].to_numpy()[:len(factors)]

axis.plot(factors, widths, marker='o', color=COLOUR['gaussian'],
          label='uniform misstatement')
axis.axvline(1.0, color=COLOUR['neutral'], lw=1.0, ls='--',
             label='correct scale')
axis.scatter([SUMMER_SIGMA_INFLATION],
             [SIGMA_SCALE_TABLE['implied width'].to_numpy()[-1]],
             color=COLOUR['heavy'], zorder=5, s=55,
             label='summer-only inflation, plotted at its factor')
axis.set_xlabel('factor applied to every stated sigma')
axis.set_ylabel('implied width of the profile peak,\n' + SCAN_PARAMETER + ' units')
axis.set_title('Figure 4. A misstated uncertainty and the confidence it implies')
axis.legend()
plt.show()

print('peak locations, to check the algebraic point above:')
print(SIGMA_SCALE_TABLE['peak'].to_string())

### 8.3 Assumption 3: errors are independent from one day to the next

**What it says.** Whether the model was too high yesterday tells you nothing about today. Each day is a fresh draw.

**Why it might be wrong here.** Two mechanisms, and only the second is serious.

1. *Measurement* error: the gap-filling in FLUXNET2015 borrows information between nearby days, so filled days are not independent of their neighbours. This is a real effect but limited to filled periods.
2. *Model* error: if DALEC2 gets the spring onset wrong, it is wrong for a run of weeks, not for isolated days. A structural error is autocorrelated almost by definition, and the likelihood treats that run of days as many independent pieces of evidence rather than one.

The consequence is the same in both cases: the effective number of independent observations is smaller than the day count, so the likelihood is sharper than the data warrants.

> **# TODO:** find a citation for autocorrelation in daily flux residuals -- either in the gap-filled product or in model-data residuals. I have not identified one I can point to, so this is currently argued from mechanism rather than from a source.

**What is demonstrated below.** Many replicate noise realisations are drawn two ways -- independent, and correlated day to day with an AR(1) coefficient -- with the same per-day standard deviation in both. *AR(1) means each day's error is a fixed fraction of yesterday's plus a fresh shock.* For each replicate the profile peak is recomputed, and the **spread of those peaks** is the honest uncertainty. It is compared against the width the likelihood itself claims.

This costs no further forward-model runs: the scan predictions are already stored, so each replicate is only arithmetic.

In [ ]:
# Replicate noise realisations: independent, and AR(1)-correlated.
def ar1_noise(rng, sigma, rho, size):
    """AR(1) noise with the stationary standard deviation equal to sigma."""
    shock = rng.normal(0.0, 1.0, size=size)
    out = np.empty(size)
    out[0] = shock[0]
    for step in range(1, size):
        out[step] = rho * out[step - 1] + np.sqrt(1.0 - rho ** 2) * shock[step]
    return out * sigma


_rng = np.random.default_rng(SEED_NOISE + 2)
peaks = {'independent': [], 'AR(1), rho = ' + format(AR1_RHO, '.2f'): []}

_started = time.perf_counter()
for _ in range(N_NOISE_REPLICATES):
    for label, noise in (
        ('independent', _rng.normal(0.0, SIGMA)),
        ('AR(1), rho = ' + format(AR1_RHO, '.2f'),
         ar1_noise(_rng, SIGMA, AR1_RHO, n_days)),
    ):
        peak, _ = profile_peak_and_width(
            SCAN_VALUES,
            gaussian_profile(TRUE_NEE + noise, SIGMA, DAY_MASK, SCAN_PREDICTIONS))
        peaks[label].append(peak)

PEAKS = {label: np.array(values) for label, values in peaks.items()}

rows = []
for label, values in PEAKS.items():
    finite = values[np.isfinite(values)]
    rows.append({
        'noise': label,
        'replicates': int(finite.size),
        'mean peak': float(finite.mean()),
        'spread of peaks (sd)': float(finite.std(ddof=1)),
        'width the likelihood claims': CLAIMED_WIDTH,
        'ratio': float(finite.std(ddof=1) / CLAIMED_WIDTH),
    })
INDEPENDENCE_TABLE = pd.DataFrame(rows).set_index('noise')
print('replicates per case:', N_NOISE_REPLICATES, ' elapsed',
      round(time.perf_counter() - _started, 1), 's')
print('true', SCAN_PARAMETER, '=', format(TRUE_SCAN_VALUE, '.6g'))
print()
INDEPENDENCE_TABLE

**Figure 5.** The spread of the likelihood's preferred value across replicate noise realisations, for independent and for correlated noise of the same size. The shaded band is the uncertainty the likelihood claims for a single dataset. If a histogram is wider than the band, the likelihood is claiming more precision than the data supports.

In [ ]:
figure, axis = plt.subplots(figsize=FIGSIZE_WIDE)
edges = np.histogram_bin_edges(
    np.concatenate([v[np.isfinite(v)] for v in PEAKS.values()]), bins=35)
for (label, values), colour in zip(PEAKS.items(), (COLOUR['gaussian'], COLOUR['heavy'])):
    axis.hist(values[np.isfinite(values)], bins=edges, alpha=0.55, color=colour,
              label=label + ' noise')
if np.isfinite(CLAIMED_WIDTH):
    axis.axvspan(TRUE_SCAN_VALUE - CLAIMED_WIDTH, TRUE_SCAN_VALUE + CLAIMED_WIDTH,
                 color=COLOUR['truth'], alpha=0.16,
                 label='width the likelihood claims (1 sd)')
axis.axvline(TRUE_SCAN_VALUE, color=COLOUR['truth'], lw=2.0, label='true value')
axis.set_xlabel('profile peak in ' + SCAN_PARAMETER + ', one per replicate')
axis.set_ylabel('replicates')
axis.set_title('Figure 5. Correlated errors and the precision the likelihood claims')
axis.legend()
plt.show()

### 8.4 Assumption 4: errors have zero mean

**What it says.** The model is not systematically above or below the observations. Errors cancel out over the record.

**Why it might be wrong here.** Any persistent structural discrepancy breaks it. Concretely at this site: the modelled growing season is currently about 41 days longer than the record indicates, with onset near day 79 against 110-111 observed. A model that starts taking up carbon six weeks early has a one-signed error every spring. That is a non-zero mean error over a substantial part of each year, and the likelihood has nowhere to put it except into the parameters.

There is also a subtler one. The synthetic data here is generated from the same model that is fitted to it, so by construction there is no structural error at all -- which is precisely why the demonstration has to *impose* a bias rather than measure one.

> **Source:** the growing-season discrepancy is measured in this project and recorded as an open finding; it is not a literature claim. **# TODO:** record the file the 41-day figure is stated in, so this can be cited rather than asserted.

**What is demonstrated below.** A constant offset is added to the synthetic observations, and the profile peak is recomputed. This shows how much of a flux bias gets absorbed into the parameter rather than showing up as a worse fit. A constant offset is the simplest possible case; a seasonal one is worse, because the parameter that best absorbs it differs by season.

In [ ]:
# A constant offset added to the observations, and where the likelihood then
# puts the parameter.
rows = []
for bias in BIAS_VALUES:
    peak, width = profile_peak_and_width(
        SCAN_VALUES,
        gaussian_profile(OBS_GAUSSIAN + bias, SIGMA, DAY_MASK, SCAN_PREDICTIONS))
    rows.append({
        'bias, g C m-2 d-1': bias,
        'peak': peak,
        'peak minus truth': peak - TRUE_SCAN_VALUE,
        'shift in units of the claimed width': (
            (peak - TRUE_SCAN_VALUE) / CLAIMED_WIDTH if np.isfinite(CLAIMED_WIDTH) else np.nan),
    })
BIAS_TABLE = pd.DataFrame(rows).set_index('bias, g C m-2 d-1')
print('true', SCAN_PARAMETER, '=', format(TRUE_SCAN_VALUE, '.6g'))
print('claimed width (1 sd) =', format(CLAIMED_WIDTH, '.4g'))
print('for scale, the median stated sigma is',
      format(float(np.median(SIGMA[DAY_MASK])), '.3f'), 'g C m-2 d-1')
print()
BIAS_TABLE

**Figure 6.** How a constant offset in the observations moves the parameter the likelihood prefers. The vertical axis is expressed in units of the uncertainty the likelihood claims, so a value of 1 means the offset has moved the estimate by a full standard deviation while the stated uncertainty stays the same.

In [ ]:
figure, axis = plt.subplots(figsize=(7.0, 4.4))
axis.plot(BIAS_TABLE.index, BIAS_TABLE['shift in units of the claimed width'],
          marker='o', color=COLOUR['heavy'])
axis.axhline(0.0, color=COLOUR['neutral'], lw=1.0)
axis.axvline(0.0, color=COLOUR['truth'], lw=1.5, label='unbiased')
axis.set_xlabel('constant offset added to the observations, g C m$^{-2}$ d$^{-1}$')
axis.set_ylabel('shift of the preferred value,\nin units of the claimed uncertainty')
axis.set_title('Figure 6. Where a systematic error goes')
axis.legend()
plt.show()

---

# 9. Misspecification experiment

The centrepiece of Part B, and the only part that runs the sampler.

**The design.** A synthetic dataset is generated with heavy-tailed noise -- so the Gaussian likelihood is wrong about the error shape, and wrong in the way section 8.1 says is most likely at a flux site. The same dataset is then fitted twice: once with the Gaussian likelihood in use, and once with a heavier-tailed likelihood. Because the true parameters are known, both fits can be scored against them.

**What is reported.**

- **Bias**: posterior mean minus true value, per parameter, both in the parameter's own units and divided by the posterior standard deviation. *In words: how far off the fit is, and whether it is far off relative to how confident it claims to be.*
- **Interval coverage**: whether the true value falls inside the stated credible interval.

**A precision point about coverage, which matters for how the table can be described.** With one dataset per likelihood, what the table reports is the **fraction of parameters** whose truth falls inside its interval. That is not the same thing as frequentist coverage, which asks how often the interval contains the truth across *repeated datasets*. Reading the first as the second would be a mistake, and it is the kind of mistake worth not making in a thesis. `N_COVERAGE_REPLICATES` in the config cell raises the number of datasets; the cost is a full pair of fits per replicate, so it is left at 1 by default.

**A second precision point, about the comparison itself.** A Student-t with $\nu$ degrees of freedom and scale $s$ has variance $s^2\nu/(\nu-2)$, not $s^2$. Handing both likelihoods the same $\sigma_d$ would therefore give the Student-t a wider error distribution as well as a heavier-tailed one, and the comparison would confound the two. `MATCH_STUDENT_T_VARIANCE` rescales the Student-t scale so that the two have equal variance and differ only in shape.

**Cost.** Two NUTS fits. For scale: this project's recorded plumbing run took about 1220 seconds for 2 chains of 200 tune plus 200 draws over 730 days. This notebook's defaults are the same sampler settings over a shorter block. Set `RUN_MISSPECIFICATION_FITS = False` for a first pass that skips both fits.

In [ ]:
# The heavy-tailed dataset this experiment is fitted to, and the two models.
import pymc as pm

from dalec.compute import ensure_configured
from dalec.likelihood import build_likelihood
from dalec.model import build_forward_graph
from dalec.priors import build_priors

FIT_OBSERVATIONS = OBS_HEAVY          # built in section 8.1
STUDENT_T_SCALE = (
    SIGMA * np.sqrt((FIT_NU - 2.0) / FIT_NU) if MATCH_STUDENT_T_VARIANCE else SIGMA)


def build_fit_model(observations, likelihood):
    """Assemble priors, forward model and one of two likelihoods.

    The Gaussian branch calls the project's own dalec.likelihood.build_likelihood
    rather than re-implementing it, so the fit being scored is the fit the
    project actually performs. The Student-t branch is written here because the
    project has no such likelihood.
    """
    block = dataclasses.replace(
        BLOCK, nee_obs=observations, nee_unc=SIGMA,
        nee_qc=np.where(DAY_MASK, 1.0, 0.0), nee_mask=DAY_MASK,
    )
    with pm.Model() as model:
        priors = build_priors(t_air=BLOCK.t_air, convention=CONVENTION)
        graph = build_forward_graph(
            parameters=priors.parameters,
            doy=BLOCK.doy.astype(float), t_air=BLOCK.t_air,
            t_max=BLOCK.t_max, t_min=BLOCK.t_min,
            sw_in=BLOCK.sw_in, co2=BLOCK.co2,
            latitude_deg=ACM.latitude_deg, coefficients=ACM.coefficients,
            frost_threshold_degc=ACM.frost_threshold_degc,
        )
        if likelihood == 'gaussian':
            build_likelihood(predicted=graph.nee, site_data=block)
        elif likelihood == 'student_t':
            index = np.flatnonzero(DAY_MASK)
            pm.StudentT('nee_obs', FIT_NU, mu=graph.nee[index],
                        sigma=STUDENT_T_SCALE[index],
                        observed=observations[index])
        else:
            raise ValueError('unknown likelihood ' + repr(likelihood))
    return model


print('fitting dataset  heavy-tailed, nu =', HEAVY_TAIL_NU)
print('days in the fit  ', int(DAY_MASK.sum()), 'of', BLOCK.n_days)
print('likelihood A     Gaussian, sigma = stated RANDUNC'
      if USE_REAL_SIGMA else 'likelihood A     Gaussian, constant sigma')
print('likelihood B     Student-t, nu =', FIT_NU, '| variance matched:',
      MATCH_STUDENT_T_VARIANCE)
print()
print('sampler          ', SAMPLE_CHAINS, 'chains,', SAMPLE_TUNE, 'tune,',
      SAMPLE_DRAWS, 'draws,', SAMPLE_CORES, 'core(s)')
print('reference cost    about 1220 s for 2 x (200 + 200) over 730 days')
print('                  (reports/sampling/plumbing_hemisurface_meta.json)')

In [ ]:
# The two fits. This is the expensive cell.
FITS = {}

if RUN_MISSPECIFICATION_FITS:
    ensure_configured()          # pins the PyTensor backend and checks it
    for label, kind in (('Gaussian', 'gaussian'), ('Student-t', 'student_t')):
        print('fitting with the', label, 'likelihood ...', flush=True)
        started = time.perf_counter()
        model = build_fit_model(FIT_OBSERVATIONS, kind)
        with model:
            FITS[label] = pm.sample(
                draws=SAMPLE_DRAWS, tune=SAMPLE_TUNE, chains=SAMPLE_CHAINS,
                cores=SAMPLE_CORES, random_seed=SEED_NOISE + 7,
                progressbar=False, idata_kwargs={'log_likelihood': False},
            )
        print('   done in', round(time.perf_counter() - started, 1), 's', flush=True)
else:
    print('RUN_MISSPECIFICATION_FITS is False, so no sampling was done.')
    print('Set it to True in the config cell to run this section.')

In [ ]:
# Convergence, before anything is read off the posteriors.
#
# round_to='none' is the ONLY value that disables rounding: ArviZ reads Python
# None as the default and falls through to two significant figures, which has
# already hidden a failed convergence check in this project once.
if FITS:
    import arviz as az

    for label, idata in FITS.items():
        summary = az.summary(idata, round_to='none')
        stats = (idata['sample_stats'].to_dataset()
                 if hasattr(idata['sample_stats'], 'to_dataset')
                 else idata['sample_stats'])
        diverging = int(stats['diverging'].to_numpy().sum()) if 'diverging' in stats else -1
        print(label.ljust(10),
              'worst r-hat', format(summary['r_hat'].max(), '.4f'),
              '| lowest bulk ESS', format(summary['ess_bulk'].min(), '.0f'),
              '| divergences', diverging)
    print()
    print('These are short runs by design. Read them as a check that the machinery')
    print('worked, not as converged posteriors.')
else:
    print('no fits to check')

In [ ]:
# Bias and interval coverage per parameter, for both likelihoods.
def score_fit(idata, true_values, level):
    """Posterior mean, bias, and whether the truth lies inside the interval."""
    posterior = (idata['posterior'].to_dataset()
                 if hasattr(idata['posterior'], 'to_dataset') else idata['posterior'])
    tail = 0.5 * (1.0 - level)
    rows = []
    for name, truth in true_values.items():
        if name not in posterior:
            continue
        samples = posterior[name].to_numpy().reshape(-1)
        low, high = np.percentile(samples, [100.0 * tail, 100.0 * (1.0 - tail)])
        sd = float(samples.std(ddof=1))
        rows.append({
            'parameter': name,
            'true': truth,
            'posterior mean': float(samples.mean()),
            'bias': float(samples.mean() - truth),
            'bias / posterior sd': float((samples.mean() - truth) / sd) if sd > 0 else np.nan,
            'interval low': float(low),
            'interval high': float(high),
            'truth inside': bool(low <= truth <= high),
        })
    return pd.DataFrame(rows).set_index('parameter')


if FITS:
    SCORES = {label: score_fit(idata, TRUE_VALUES, HDI_PROB)
              for label, idata in FITS.items()}
    COMPARISON = pd.concat(
        {label: frame[['bias', 'bias / posterior sd', 'truth inside']]
         for label, frame in SCORES.items()}, axis=1)
    print('credible level', HDI_PROB, '| equal-tailed intervals')
    print('parameters scored:', len(next(iter(SCORES.values()))))
    print()
    for label, frame in SCORES.items():
        print(label.ljust(10),
              'parameters with the truth inside the interval:',
              int(frame['truth inside'].sum()), 'of', len(frame),
              '| median |bias / sd|',
              format(frame['bias / posterior sd'].abs().median(), '.3f'))
    print()
    COMPARISON
else:
    print('no fits to score')

**Figure 7.** Recovery of the known parameters under the two likelihoods, on the same heavy-tailed dataset. Each parameter is one row. The horizontal axis is bias in units of the posterior standard deviation, so zero is perfect recovery and the shaded band marks plus or minus one. A filled marker means the true value fell inside the credible interval; an open marker means it did not.

*What the two axes trade off:* a point near zero on a narrow posterior is good recovery; a point near zero on a very wide posterior is an uninformative fit that cannot be wrong. The marker fill carries that information, the position does not.

In [ ]:
if FITS:
    order = SCORES['Gaussian']['bias / posterior sd'].abs().sort_values().index
    figure, axis = plt.subplots(figsize=(8.5, 0.32 * len(order) + 2.2))
    offsets = {'Gaussian': -0.16, 'Student-t': 0.16}
    colours = {'Gaussian': COLOUR['gaussian'], 'Student-t': COLOUR['heavy']}

    axis.axvspan(-1.0, 1.0, color=COLOUR['neutral'], alpha=0.07,
                 label='plus or minus one posterior sd')
    axis.axvline(0.0, color=COLOUR['truth'], lw=1.6, label='no bias')
    for label, frame in SCORES.items():
        frame = frame.loc[order]
        positions = np.arange(len(order)) + offsets[label]
        inside = frame['truth inside'].to_numpy()
        values = frame['bias / posterior sd'].to_numpy()
        axis.scatter(values[inside], positions[inside], color=colours[label],
                     s=42, label=label + ', truth inside the interval')
        axis.scatter(values[~inside], positions[~inside], facecolors='none',
                     edgecolors=colours[label], s=42,
                     label=label + ', truth outside')
    axis.set_yticks(range(len(order)))
    axis.set_yticklabels(order)
    axis.set_xlabel('bias in units of the posterior standard deviation')
    axis.set_title('Figure 7. Parameter recovery under two likelihoods, '
                   'heavy-tailed data')
    axis.legend(loc='lower right', fontsize=7)
    plt.show()
else:
    print('Figure 7 not drawn: RUN_MISSPECIFICATION_FITS is False.')

**How to read Figure 7 and the table above.** The readings available, none of which this notebook makes for you:

- One likelihood recovers more parameters inside their intervals than the other.
- One has systematically smaller bias in posterior-sd units.
- The two are indistinguishable, which would say that at this noise level and this record length the error shape does not matter much -- a result about this configuration, not a general one.
- A parameter is badly recovered under *both*, which points at identifiability rather than at the likelihood.

**Raising `N_COVERAGE_REPLICATES`** turns the coverage column into something closer to real coverage, at a cost of one pair of fits per replicate.

---

# 10. Where the information actually comes from

Not every day contributes equally. Two things decide a day's influence:

$$w_d \;\propto\; \frac{1}{\sigma_d^{2}} \qquad\text{and}\qquad I_d \;\propto\; \frac{1}{\sigma_d^{2}}\left(\frac{\partial m_d}{\partial \theta}\right)^{2}$$

*In words: the first says a day counts for more when its stated uncertainty is small. The second adds that it also has to matter -- a day the parameter barely changes carries no information about that parameter however precisely it was measured.*

The second quantity is the **Fisher information**: how sharply the likelihood responds to the parameter on that day. It is computed here by finite difference from the scan already stored, so it costs nothing extra.

**The plain-English warning this section exists for:** a small number of days can dominate the fit if their stated uncertainty is small. If those days are concentrated in one season, the calibration is effectively a calibration against that season, whatever the record length suggests.

**Scope note.** The weights below are built from the uncertainty column and from model sensitivity. No observed flux value enters, and under `USE_REAL_SIGMA = False` the precision weights become uniform by construction.

In [ ]:
# Per-day weights, and the seasonal balance of the fit.
SEASON_OF_MONTH = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM',
                   6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}
SEASON_ORDER = ['DJF', 'MAM', 'JJA', 'SON']
seasons = np.array([SEASON_OF_MONTH[m] for m in BLOCK_MONTHS])

precision = np.where(DAY_MASK, 1.0 / SIGMA ** 2, 0.0)

# Model sensitivity by central difference across the scan grid, evaluated at
# the grid point nearest the true value.
nearest = int(np.argmin(np.abs(SCAN_VALUES - TRUE_SCAN_VALUE)))
low = max(0, nearest - 1)
high = min(SCAN_POINTS - 1, nearest + 1)
sensitivity = ((SCAN_PREDICTIONS[high] - SCAN_PREDICTIONS[low])
               / (SCAN_VALUES[high] - SCAN_VALUES[low]))
information = np.where(DAY_MASK, (sensitivity ** 2) / SIGMA ** 2, 0.0)

WEIGHTS = pd.DataFrame({
    'season': seasons,
    'contributes': DAY_MASK,
    'precision weight': precision / precision.sum(),
    'information weight': (information / information.sum()
                           if information.sum() > 0 else information),
}, index=BLOCK_DATES)

SEASON_TABLE = WEIGHTS.groupby('season').agg(
    days_in_block=('contributes', 'size'),
    days_contributing=('contributes', 'sum'),
    precision_share=('precision weight', 'sum'),
    information_share=('information weight', 'sum'),
).reindex(SEASON_ORDER)
SEASON_TABLE['precision_share'] = (100 * SEASON_TABLE['precision_share']).round(1)
SEASON_TABLE['information_share'] = (100 * SEASON_TABLE['information_share']).round(1)

print('shares are percentages of the total over the block')
print('sensitivity is d(NEE)/d(' + SCAN_PARAMETER + ') at the true value')
print()
SEASON_TABLE

In [ ]:
# How concentrated is the fit? The share of total weight held by the most
# heavily weighted days.
def concentration(weights):
    ordered = np.sort(weights[weights > 0])[::-1]
    if ordered.size == 0:
        return {}
    cumulative = np.cumsum(ordered) / ordered.sum()
    out = {}
    for fraction in (0.05, 0.10, 0.25, 0.50):
        count = max(1, int(round(fraction * ordered.size)))
        out['top ' + format(100 * fraction, '.0f') + ' per cent of days'] = round(
            100.0 * cumulative[count - 1], 1)
    return out


CONCENTRATION = pd.DataFrame({
    'precision weight': concentration(WEIGHTS['precision weight'].to_numpy()),
    'information weight': concentration(WEIGHTS['information weight'].to_numpy()),
})
print('percentage of the total weight held by the most heavily weighted days')
print()
CONCENTRATION

**Figure 8.** Where the fit's weight sits across the year. The top panel shows each contributing day's share of the total, on the two definitions above. The bottom panel is the cumulative share against the number of days, sorted from the heaviest day down: a curve that rises steeply means a few days carry the fit, and the diagonal is what perfectly even weighting would look like.

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(10.5, 7.0),
                            gridspec_kw={'height_ratios': [2, 1.5]})

contributing = WEIGHTS[WEIGHTS['contributes']]
axes[0].plot(contributing.index, 100 * contributing['precision weight'],
             color=COLOUR['gaussian'], lw=1.0, label='precision weight, 1 / sigma squared')
axes[0].plot(contributing.index, 100 * contributing['information weight'],
             color=COLOUR['heavy'], lw=1.0,
             label='information weight, sensitivity squared / sigma squared')
axes[0].set_ylabel('share of the total, per cent')
axes[0].set_title('Figure 8. Where the likelihood gets its information')
axes[0].legend()

n_positive = 0
for column, colour in (('precision weight', COLOUR['gaussian']),
                       ('information weight', COLOUR['heavy'])):
    values = np.sort(WEIGHTS[column].to_numpy()[WEIGHTS[column] > 0])[::-1]
    if values.size:
        share = 100 * np.cumsum(values) / values.sum()
        axes[1].plot(np.arange(1, values.size + 1), share, color=colour,
                     label=column)
        n_positive = max(n_positive, values.size)
if n_positive:
    axes[1].plot([0, n_positive], [0, 100], color=COLOUR['neutral'], lw=0.8,
                 ls='--', label='perfectly even weighting')
axes[1].set_xlabel('number of days, heaviest first')
axes[1].set_ylabel('cumulative share of\ntotal weight, per cent')
axes[1].legend(loc='lower right')
plt.show()

---

# 11. Menu of likelihood alternatives

Options with costs, in the same form as section 6. None is a recommendation.

| Alternative | What it fixes | What it costs in sampling | Can it be justified from the literature? |
|---|---|---|---|
| **Heavier-tailed error distribution** (Student-t with fixed $\nu$, or a double-exponential) | Assumption 1. A single bad day stops dominating the fit, because the likelihood no longer regards it as nearly impossible. | Mild. One extra shape parameter if $\nu$ is estimated rather than fixed, and $\nu$ is often weakly identified, which can slow mixing. The gradient is as smooth as the Gaussian's. | Partly. A double-exponential error shape for tower fluxes is a standard claim -- see the UNVERIFIED candidates in 8.1. Choosing $\nu$ specifically would be my own choice and would need stating. |
| **Estimate a scaling factor on the stated uncertainty** ($\sigma_d \to \alpha\sigma_d$ with $\alpha$ inferred) | Assumption 2. Lets the data say whether the stated uncertainties are the right size, instead of assuming it. | Low to moderate. One extra parameter, but it correlates with anything that controls the overall fit quality, and that correlation can be stiff. Needs its own prior. | Yes in general terms -- estimating an error-scale parameter is standard Bayesian practice. Its interpretation here is not: $\alpha > 1$ would mostly be measuring model error, not measurement error. |
| **Correlated errors across days** (AR(1) or a Gaussian process on the residual) | Assumption 3. Stops a run of correlated days being counted as many independent pieces of evidence. Section 8.3 measures what that is worth. | High. The log-likelihood is no longer a sum of independent terms; it needs a covariance matrix that is either solved or given special structure. For a multi-year daily record that is a large object, and it must stay differentiable for NUTS. | Plausible in principle, but I have no citation for the daily flux case yet -- see the TODO in 8.3. It would need defending as my own choice. |
| **Seasonal or per-day weighting** (down-weighting a season, or days of a given quality class) | Assumptions 2 and 4 together, where the discrepancy is known to be concentrated -- the long modelled growing season being the obvious case here. | Low mechanically: it is a change of $\sigma_d$. | This is the weakest of the four. Weights chosen because the model fits badly there are a way of hiding a structural error rather than representing it, and the project's design decisions already exclude seasonal filtering of the assimilation target. It would need a strong independent reason. |
| **Keep the Gaussian and report the assumption as a limitation** | Nothing. | Nothing. | Defensible, and it is what the current design does. The cost is that any statement about posterior *width* rests on an assumption the notebook has shown may be wrong. |

---

# 12. Talking points for the meeting

### What this notebook supports

1. **The priors are traceable.** Every bound in section 4 resolves to a recorded source, and section 3 shows what each revision responded to. Of the 26 rows, the provenance table says exactly how many rest on a site measurement and how many do not.
2. **The priors have been checked before fitting.** The prior predictive check in section 5 runs the priors through the forward model and scores the result against named physical rules, each with a stated justification. This check has already changed the priors once, at v1 to v2.
3. **The likelihood's assumptions are stated and their cost is measurable.** Section 8 shows, on synthetic data where the truth is known, what each of the four assumptions does to the estimate and to the claimed uncertainty when it fails.
4. **The cost of the assumption most likely to be wrong has been measured directly.** Section 9 fits the same misspecified dataset with two likelihoods and scores both against the known truth.

### Questions I would like your judgement on

Each can be answered yes or no, or by choosing between the named options. Space is left underneath.

**Q1. The annual-NEE plausibility bound has no source, because the obvious source -- the site's own measured annual NEE -- is an observed flux and this notebook excludes those by design. Should I (a) keep it as a stated assumption, (b) relax the synthetic-only rule for this one bound, or (c) drop the rule?**

>
>

**Q2. `f_som` is labelled `guess` in the prior table. Is that acceptable to leave in the thesis as a declared judgement, yes or no?**

>
>

**Q3. The 3-5 year needle longevity sets both `c_lf` and `lma` and currently has no citation. Should I (a) find and verify a source, (b) replace it with a measurement I can point to at SMEAR II, or (c) widen both priors until it no longer matters?**

>
>

**Q4. Is the Gaussian likelihood acceptable as the thesis's primary likelihood, with the misspecification experiment of section 9 reported as a sensitivity, yes or no? If no, which alternative from section 11?**

>
>

**Q5. Section 8 argues that the error term is really model error, not measurement error, and that assumptions 3 and 4 are much weaker for model error. Should the thesis (a) state this and keep the likelihood, (b) estimate a scale factor on the uncertainty, or (c) attempt a correlated-error structure?**

>
>

**Q6. The true parameter set in this notebook is drawn from the prior, which makes the prior correct by construction and isolates the likelihood. Is that the right design for the thesis chapter, or should a second experiment draw the truth from outside the prior, yes or no?**

>
>

**Q7. Is `reports/` the right place for the figures this notebook produces, or should the thesis chapter carry its own figure directory? (Currently the notebook writes nothing.)**

>
>

---

# 13. Self-critique: what is fragile in this notebook

Written before the notebook is accepted, and deliberately including the places a sharp reader would push first.

## Where a sharp reader would push first

**1. The truth is drawn from the prior, so Part B cannot see prior misspecification at all.** Every recovery result in section 9 is conditional on a prior that is correct by construction. This is the right design for isolating the likelihood and it is stated in section 5, but it means the phrase "the method recovers the parameters" is doing less work than it appears to. Question 6 in section 12 exists for this.

**2. The synthetic data is generated by the same model that is fitted to it.** There is no structural error anywhere in Part B except the one section 8.4 imposes by hand. Every real-world concern raised in section 8 -- the missing hydrology, the long growing season, the soil frost -- is absent from the experiment that is supposed to measure their cost. The demonstrations show what a *given* violation costs, not how large the violation actually is at FI-Hyy.

**3. The profile likelihood is not a posterior, and the "implied width" is not a posterior standard deviation.** Sections 8.1 to 8.4 hold 23 parameters at their true values and vary one. Real uncertainty in that parameter is larger, because the others are unknown too, and correlations between them can change the answer's direction, not just its size. The word "width" is used throughout rather than "uncertainty" for this reason, but the distinction is easy to lose when reading the figures quickly.

**4. Coverage in section 9 is across parameters, not across datasets.** Stated in the section, but it is exactly the kind of thing that gets restated loosely one draft later. With `N_COVERAGE_REPLICATES = 1` the table cannot support a sentence of the form "the interval contained the truth 85 per cent of the time".

**5. Section 9 compares two likelihoods on one dataset and one noise realisation.** Whichever comes out ahead may have come out ahead by luck. The honest version needs replicates, which is a config flag and a large compute bill.

## Things that could be silently wrong

**6. The prior draws are not nested.** Each parameter is drawn as one vector of length `N_PRIOR_DRAWS`, so raising it changes every draw rather than adding to them. A failure rate that moves between 200 and 1000 draws is not evidence of anything.

**7. `sample_reparameterised_parameters` fails for the whole batch, not per draw.** If any single draw drives the derived wood carbon non-positive -- which happens when `theta_roo` reaches the slow end of its range -- the call raises and no draws are produced at all. The truth draw in section 5 catches this and tells you to change the seed; the prior-draw cell does not, and would simply fail.

**8. `except Exception` in the prior run loop will swallow a real bug.** It records the exception type and message, which is the mitigation, but a typo producing a `TypeError` on every draw would appear as a 100 per cent failure rate rather than as a crash. If the failure rate is high, read `prior_run['error']` before reading anything else.

**9. The plausibility thresholds are mine, not measurements.** `LAI_CEILING_MULTIPLE = 2.0` is a round number I chose; the site LAI it multiplies is sourced, the multiple is not. The annual-NEE window is a stated assumption with no source at all. Either can be set until any prior fails or none does.

**10. The per-parameter attribution in section 5d is univariate and will miss joint causes.** This is not a hypothetical limitation: the v1-to-v2 revision in section 3 is exactly a joint cause, and this diagnostic would not have found it.

**11. The scan grid is coarse and the width estimate depends on it.** `profile_peak_and_width` fits a parabola to `window = 9` points near the maximum. If the profile is not close to quadratic there -- which happens when the peak is near a prior bound, or when the scan is too wide -- the fitted width is a poor summary and the function will happily return a number anyway. It returns NaN only when the fitted curvature has the wrong sign.

**12. The AR(1) demonstration reuses the same stored predictions for every replicate.** That is what makes it cheap, and it also means the replicates differ only in noise, not in anything else. That is the correct experiment for the question asked, but it is not a full Monte Carlo of the fitting procedure.

**13. `information weight` in section 10 is a one-parameter finite difference.** It is the sensitivity of NEE to `SCAN_PARAMETER` alone, evaluated at the truth, by central difference across two grid points. A different parameter would give a different seasonal picture, and a coarse grid makes the difference inaccurate. The precision weight has no such dependence.

**14. The Student-t variance matching changes what `sigma` means between the two fits.** With `MATCH_STUDENT_T_VARIANCE = True` the Student-t receives a *smaller* scale than the Gaussian so that the two variances agree. This is the right comparison for isolating tail shape, and it means the two likelihoods are not being handed the same numbers. Turning the flag off makes them equal in scale and unequal in variance. Either is defensible; they answer different questions.

**15. `nee_qc` is fabricated on the fitted block.** `build_fit_model` sets it to 1.0 on masked-in days and 0.0 elsewhere, because `SiteData` requires the field and the synthetic data has no quality flag. Nothing downstream reads it -- the mask is passed separately -- but it is a fabricated column and would be wrong to report.

## Assumptions stated but not verified

**16. Two of the three citations in section 8.1 are unverified candidates.** They are labelled as such and carry a TODO. They should not be cited in the thesis in their current state.

**17. The autocorrelation argument in 8.3 rests on mechanism, not on a source.** I could not point to a citation for autocorrelation in daily flux residuals, so the section says so rather than implying one exists.

**18. Provenance categories are my readings of `DECISIONS.md`.** Three are arguable and are flagged after the prior table: `ceff`, `theta_lit`, `theta_som`.

## Design choices in the notebook itself

**19. Section 9 is the only part that samples, and it is fragile in a notebook.** `SAMPLE_CORES = 1` is set because multi-process sampling from inside a Jupyter kernel on Windows is unreliable. That makes the chains sequential and the cell slow. `ensure_configured()` will also raise if PyTensor cannot resolve the pinned Numba backend, which is a real possibility on a machine without it.

**20. Nothing here is written to disk.** Every figure is drawn inline and lost on kernel restart. That is deliberate -- the notebook is an argument, not a build step -- but it means the figures cannot be dropped into the thesis without a save step being added.

**21. The scope guard checks one field.** `assert_no_observed_flux` checks that `nee_obs` carries no finite values on the block this notebook uses. It does not and cannot prove that no observed quantity reaches a figure by another route. The partitioned GPP and RECO products are dropped explicitly when the block is built, for that reason. `RANDUNC`, the quality mask, and the drivers are all still real, by the deliberate design of section 1.